In [1]:
import re
from typing import List
from collections import defaultdict

import polars as pl

from novami.io.file import write_pl

In [2]:
def latex_2_df(table: str, columns: List[str] = None):
    table = re.sub(
        pattern=r"\\cmidrule\{\d+\-\d+\}|\\newpage",
        string=table,
        repl=""
    )

    lines = [line.strip().rstrip("\\") for line in table.split("\n")]
    lines = [line for line in lines if line not in ["", "%"]]

    dc = defaultdict(list)
    for idx, line in enumerate(lines):
        items = [item.strip() for item in line.split("&")]
        if len(items) != 10:
            print(line)
        for key, item in zip(columns, items):
            dc[key].append(item)

    df = pl.DataFrame(dc)
    df = df.with_columns(
        pl.all().replace("", None)
    )

    df = df.with_columns(
        pl.all().forward_fill()
    )

    return df


def strip_polyrows(df: pl.DataFrame, columns: List[str]):

    def strip_string(string: str):
        string = string.strip()
        if re.match(r"\A\\\w+\{", string) is None:
            return string

        string = re.sub(r"\A\\\w+\{", "", string)[:-1]
        return string

    df = df.with_columns([
        pl.col(col).map_elements(
            strip_string,
            return_dtype=pl.String
        ).alias(col) for col in columns
    ])

    return df

def strip_values(df: pl.DataFrame, columns: List[str] = None):

    def strip_string(string: str):
        string = string.strip()
        if "\\" not in string:
            return string

        string = re.sub(r"\\\w+", "", string)
        return string

    df = df.with_columns([
        pl.col(col).map_elements(
            strip_string,
            return_dtype=pl.String
        ).alias(col) for col in columns
    ])

    return df

# Tables

In [24]:
table_s1 = r"""
    Cavalli (2002) \cite{Cavalli2002}         & PH, Reg      & PLS                                             & 3D Shape-based & $hERG$ \\
    \cmidrule{2-5}
    Ekins (2002) \cite{Ekins2002}             & PH           & Alignment-based                                 & 3D Shape-based (CATALYST) & $hERG$ \\
    \cmidrule{2-5}
    Roche (2002) \cite{Roche2002}             & BC           & SOM, PCA, PLS, NN                               & MD (VolSurf, DRAGON) & $hERG$\\
    \cmidrule{2-5}
    Ekins (2003) \cite{Ekins2003}             & PH           & Alignment-based                                 & 3D Shape-based (CATALYST) & $hERG$ \\
    \cmidrule{2-5}
    Keserü (2003) \cite{Keser2003}            & Reg          & LR + hQSAR                                      & MD (VolSurf, Sybyl) & $hERG$\\
    \cmidrule{2-5}
    Pearlstein (2003) \cite{Pearlstein2003}   & PH           & 3D Shape-based                                  & Pharmacophore & $hERG$ \\
    \cmidrule{2-5}
    Aptula (2004) \cite{Aptula2004}           & Reg          & MLR                                             & MD, Quantum Chemical & $hERG$ \\
    \cmidrule{2-5}
    Aronov (2004) \cite{Aronov2004}           & BC           & DT                                              & Pharmacophore-based & $hERG$ \\
    \cmidrule{2-5}
    Bains (2004) \cite{Bains2004}             & BC           & DT                                              & MD, Fragment-based  & $hERG$ \\
    \cmidrule{2-5}
    Fioravanzo (2004) \cite{Fioravanzo2004}   & BC           & PLS                                             & MD (DRAGON), EVA (from IR and Raman spectra), DRAGON & $hERG$ \\
    \cmidrule{2-5}
    Cianchetta (2005) \cite{Cianchetta2005}   & Reg          & PLS + hQSAR                                     & 3D Shape-based & $hERG$ \\
    \cmidrule{2-5}
    O'Brien (2005) \cite{OBrien2005}          & BC           & NN, NB                                          & Topological, Fragment-based, Circular & $hERG$ \\
    \cmidrule{2-5}
    Tobita (2005) \cite{Tobita2005}           & BC           & SVM                                             & MD (MOE), MACCS & $hERG$ \\
    \cmidrule{2-5}
    Aronov (2006) \cite{Aronov2006}           & PH           & Alignment-based                                 & 3D Shape-based (MOE) & $hERG$ \\
    \cmidrule{2-5}
    Coi (2006) \cite{Coi2006}                 & Reg          & LR                                              & MD (CODESSA) & $hERG$ \\
    \cmidrule{2-5}
    Dubus (2006) \cite{Dubus2006}             & MC           & DT                                              & MD (MOE) & $hERG$ \\
    \cmidrule{2-5}
    Ekins (2006) \cite{Ekins2006}             & Reg          & DT, SOM, Sammon Maps                            & MD (Smart Mining) & $hERG$ \\
    \cmidrule{2-5}
    Gepp (2006) \cite{Gepp2006}               & BC           & DT                                              & MD, QC (VAMP), Custom SMARTS & $hERG$ \\
    \cmidrule{2-5}
    Seierstad (2006) \cite{Seierstad2006}     & Reg          & MLP                                             & MD (MOE), Topological (KH, GC), Atom Pairs, Fragment (ISIDA) & $hERG$ \\
    \cmidrule{2-5}
    Sun (2006) \cite{Sun2006}                 & BC           & NB                                              & Atom Types, Circular & $hERG$ \\
    \cmidrule{2-5}
    Song (2006) \cite{Song2006}               & Reg          & SVM, PLS, RF                                    & Fragment (Clark 2005) & $hERG$ \\
    \cmidrule{2-5}
    Yoshida (2006) \cite{Yoshida2006}         & Reg          & LR                                              & MD & $hERG$ \\
    \cmidrule{2-5}
    Du (2007) \cite{Du2007}                   & Reg          & PLS                                             & Docking scores & $hERG$ \\
    \cmidrule{2-5}
    Leong (2007) \cite{Leong2007}             & Reg          & SVM                                             & Pharmacophore & $hERG$ \\
    \cmidrule{2-5}
    Obrezanova (2007) \cite{Obrezanova2007}   & Reg          & GP                                              & MD (in-house), custom SMARTS & $hERG$ \\
    \cmidrule{2-5}
    Chekmarev (2008) \cite{Chekmarev2008}     & BC           & kNN, SVM, SOM                                   & 3D Shape-based & $hERG$ \\
    \cmidrule{2-5}
    Gunturi (2008) \cite{Gunturi2008}         & Reg          & kNN, LOESS                                      & MD (BioSuite, SYBYL) & $hERG$ \\
    %\cmidrule{2-5}
    \newpage
    Jia (2008) \cite{Jia2008}                 & BC           & SVM                                             & Atom Types & $hERG$ \\
    \cmidrule{2-5}
    Kramer (2008) \cite{Kramer2008}           & Reg          & MLR, PLS, SVR                                   & MD and QC & $hERG$ \\
    \cmidrule{2-5}
    Li (2008) \cite{Li2008}                   & BC           & SVM                                             & Field-based & $hERG$ \\
    \cmidrule{2-5}
    Thai\sep (2008) \cite{Thai2008BMC}        & BC           & NB                                              & MD & $hERG$ \\
    \cmidrule{2-5}
    Thai\sep (2008) \cite{Thai2008_CBDD}      & BC, Reg      & PLS, CP-NN                                      & MD (MOE) & $hERG$ \\
    \cmidrule{2-5}
    Wang (2008) \cite{Wang2008}               & BC           & DT, MLP, RBFNetwork, SVM, LR                    & 3D MD, QC & $hERG$ \\
    \cmidrule{2-5}
    Coi (2009) \cite{Coi2009}                 & Reg          & MLR                                             & MD (CODESSA), QC (MOPAC) & $hERG$ \\
    \cmidrule{2-5}
    Ermondi (2009) \cite{Ermondi2009}         & Reg          & PLS                                             & Field-based & $hERG$ \\
    \cmidrule{2-5}
    Hansen (2009) \cite{Hansen2009}           & Reg          & RF, SVR, GP, RidgeReg                           & MD (MOE), Field-based, Pharmacophore & $hERG$ \\
    \cmidrule{2-5}
    Nisius\sep (2009) \cite{Nisius2009CBDD}   & BC           & SVM                                             & Fragment & $hERG$ \\
    \cmidrule{2-5}
    Nisius\sep (2009) \cite{Nisius2009JCIM}   & BC           & kNN                                             & Circular, Fragment, 3D-shape based & $hERG$ \\
    \cmidrule{2-5}
    Thai (2009) \cite{Thai2009}               & BC           & PLS, NB, CP-NN                                  & MD (MOE, VolSurf), SIBAR & $hERG$ \\
    \cmidrule{2-5}
    Doddareddy (2010) \cite{Doddareddy2010}   & BC           & LDA, SVM                                        & Circular & $hERG$ \\
    \cmidrule{2-5}
    Obrezanova (2010) \cite{Obrezanova2010}   & BC           & GP, DT, SVM, RF, PLS                            & MD (in-house), custom SMARTS & $hERG$ \\
    \cmidrule{2-5}
    Su (2010) \cite{Su2010}                   & BC, Reg      & PLS                                             & 4D-FP (MDDM), MD (MOE) & $hERG$ \\
    \cmidrule{2-5}
    Wiśniowska (2010) \cite{Wisniowska2010}   & BC           & RF                                              & MD, Circular, Experimental & $hERG$ \\
    \cmidrule{2-5}
    Cuny (2011) \cite{Cuny2011}               & BC, Reg      & kNN, PLS                                        & MD (MOE), Pharmacophore & $hERG$ \\
    \cmidrule{2-5}
    Kim (2011) \cite{Kim2011}                 & BC           & NB, RF                                          & MD, Circular & $hERG$ \\
    \cmidrule{2-5}
    Obiol-Pardo (2011) \cite{ObiolPardo2011}  & Reg          & PLS                                             & Field-based, Docking Scores & $hERG$, $K_v7.1$ \\
    \cmidrule{2-5}
    Robinson (2011) \cite{Robinson2011}       & BC           & SVM, RF                                         & MD (MOE), Circular & $hERG$ \\
    \cmidrule{2-5}
    Sinha (2011) \cite{Sinha2011}             & Reg          & MLR                                             & MD & $hERG$ \\
    \cmidrule{2-5}
    Shen (2011) \cite{Shen2011}               & BC           & SVM                                             & 4D-FP (MDDM), MD (MOE, VolSurf) & $hERG$ \\
    \cmidrule{2-5}
    Broccatelli (2012) \cite{Broccatelli2012} & BC           & PLS, LDA, SVM, RF, kNN                          & MD (MOE, DRAGON, CDK), Field-based (VS+), Interaction (FLAP), Fragment & $hERG$ \\
    \cmidrule{2-5}
    Kar (2012) \cite{Kar2012}                 & BC, Reg      & LDA, PLS                                        & MD (PaDEL, Cerius2, DRAGON) & $hERG$ \\
    \cmidrule{2-5}
    Polak (2012) \cite{Polak2012}             & Reg          & MLP, NFS, SMO, RF                               & MD (ChemAxon) & $K_v7.1$ \\
    \cmidrule{2-5}
    Tan (2012) \cite{Tan2012}                 & BC, Reg      & PLS                                             & MD (CODESSA), QC (MOPAC), Pharmacophore & $hERG$ \\
    \cmidrule{2-5}
    Wang (2012) \cite{Wang2012}               & BC           & NB                                              & Circular, Path-based & $hERG$ \\
    \cmidrule{2-5}
    Wiśniowska (2012) \cite{Winiowska2012}    & Reg          & MLP, MON-MLP, NFS                               & MD (ChemAxon) & $Ca_v1.2$ \\
    \cmidrule{2-5}
    Czodrowski (2013) \cite{Czodrowski2013}   & BC           & RF                                              & MD (RDKit) & $hERG$ \\
    \cmidrule{2-5}
    Kireeva (2013) \cite{Kireeva2013}         & BC           & GTM, SVM                                        & Fragment (IPLF), Path-based (CDK) & $hERG$ \\
    \cmidrule{2-5}
    Braga (2014) \cite{Braga2014}             & BC           & SVM, RF, GBM, DT                                & Circular, Fragment (MACCS, PubChem), Pharmacophore & $hERG$ \\
    \newpage
    Kratz (2014) \cite{Kratz2014}             & BC           & Alignment-based                                 & Pharmacophore (LigandScout, StudioCatalyst) & $hERG$ \\
    \cmidrule{2-5}
    Liu (2014) \cite{Liu2014}                 & BC           & NB                                              & MD, Circular & $hERG$ \\
    \cmidrule{2-5}
    Braga (2015) \cite{Braga2015}             & BC, MC       & SVM                                             & Circular, Path-based (CDK) & $hERG$ \\
    \cmidrule{2-5}
    Du (2015) \cite{Du2015}                   & BC           & Winnow, SVM                                     & MD, Circular & $hERG$ \\
    \cmidrule{2-5}
    Ma (2015) \cite{Ma2015}                   & Reg          & RF, SVM, DNN, GBM, GP                           & MD, Circular, Fragment (MACCS), AtomPairs & $hERG$, $Na_v1.5$, $Ca_v1.2$ \\
    \cmidrule{2-5}
    Chavan (2016) \cite{Chavan2016}           & BC           & kNN                                             & Path-based (CDK), Topological (EState), Fragment (MACCS, PubChem, SubFP) & $hERG$ \\
    \cmidrule{2-5}
    Didziapetris (2016) \cite{Didzia2016}     & BC           & GBM                                             & Topological, Physicochemical & $hERG$ \\
    \cmidrule{2-5}
    Gobbi (2016) \cite{Gobbi2016}             & Reg          & LR                                              & String-fragment & $hERG$ \\
    \cmidrule{2-5}
    Sheridan (2016) \cite{Sheridan2016}       & Reg          & XGB                                             & Atom Pairs, Donor-Acceptor Pairs & $hERG$, $Na_v1.5$, $Ca_v1.2$ \\
    \cmidrule{2-5}
    Wang (2016) \cite{Wang2016}               & BC           & NB, SVM                                         & Pharmacophore (DS2.5) & $hERG$ \\
    \cmidrule{2-5}
    Zhang (2016) \cite{Zhang2016}             & BC           & SVM, NB, kNN, RF, DT                            & MD (PaDEL), Path-based (CDK), Fragment (MACCS, PubChem, SubFP) & $hERG$ \\
    \cmidrule{2-5}
    Chemi (2017) \cite{Chemi2017}             & PH, Reg      & PLS                                             & Pharmacophore & $hERG$ \\
    \cmidrule{2-5}
    Li (2017) \cite{Li2017}                   & BC           & kNN, SVM, MLR, PLS, RF, DT                      & MD (DRAGON, ChemAxon), Path-based (CDK), Topological (EState), Fragment (ISIDA, SA, GSFrag, SiRMS), 3D (Inductive, Mera and Mersy, Adriana, Spectrophores), String-based (QNPR) & $hERG$ \\
    \cmidrule{2-5}
    Sun (2017) \cite{Sun2017}                 & BC           & SVC                                             & Atom Types & $hERG$ \\
    \cmidrule{2-5}
    Alves (2018) \cite{Alves2018}             & BC           & kNN                                             & MD (PaDEL, DRAGON), Circular, Fragment (SiRMS) & $hERG$ \\
    \cmidrule{2-5}
    Munawar (2018) \cite{Munawar2018}         & Reg          & PLS                                             & Field-based (GRIND) & $hERG$ \\
    \cmidrule{2-5}
    Siramshetty (2018) \cite{Siramshetty2018} & BC           & kNN, SVM, RF                                    & Circular, Fragment (MACCS, PubChem) & $hERG$ \\
    \cmidrule{2-5}
    Wacker (2018) \cite{Wacker2018}           & BC, Reg      & XGB, RF                                         & MD (RDKit), Pharmacophore &  $hERG$ \\
    \cmidrule{2-5}
    Yang (2018) \cite{Yang2018}               & N/A          & RF, SVM, kNN                                    & Circular, Fragment (MACCS), Atom Pairs &  $hERG$ \\
    \cmidrule{2-5}
    Cai (2019) \cite{Cai2019}                 & BC           & DNN, NB, SVM, RF, GCN                           & MD (MOE), Embedding (Mol2Vec) & $hERG$ \\
    \cmidrule{2-5}
    Hu (2019) \cite{Hu2019}                   & BC           & GCN, RF                                         & Circular, Graphs & $hERG$ \\
    \cmidrule{2-5}
    Konda (2019) \cite{Konda2019}             & BC           & RF, MLP, SMO                                    & MD (PaDEL) & $hERG$ \\
    \cmidrule{2-5}
    Lee (2019) \cite{Lee2019}                 & BC           & LR, LogReg, Ridge, NN, NB, RF                   & MD (DRAGON), Circular & $hERG$ \\
    \cmidrule{2-5}
    Munawar (2019) \cite{Munawar2019}         & Reg          & PLS                                             & Interaction-based (PLIF) & $hERG$ \\
    \cmidrule{2-5}
    Negami (2019) \cite{Negami2019}           & Reg          & LR                                              & FEP-based (MP-CAFEE) & $hERG$ \\
    \cmidrule{2-5}
    Ogura (2019) \cite{Ogura2019}             & BC           & SVM                                             & MD (MOE, PipelinePilot), Circular & $hERG$ \\
    \cmidrule{2-5}
    Zhang (2019) \cite{Zhang2019}             & BC           & RF, DNN                                         & MD (ChemoPy, MOE, PaDEL) & $hERG$ \\
    \cmidrule{2-5}
    Choi (2020) \cite{Choi2020}               & BC           & ANN, NB, SVM, RF                                & Circular, Path-based (CDK) & $hERG$ \\
    \newpage
    Khalifa (2020) \cite{Khalifa2020}         & BC, Reg      & BayesNet, NB, MLP, K-Star, RF, SVM, SGD, SMO, GP, GLM, GBM, FFNN & Path-based (CDK), Topological (EState), Fragment (MACCS, PubChem, SubFP, Klek), Atom Pairs & $Na_v1.5$ \\
    \cmidrule{2-5}
    Kim (2020) \cite{Kim2020}                 & BC           & Self-attention DNN                              & Circular & $hERG$ \\
    \cmidrule{2-5}
    Liu (2020) \cite{Liu2020}                 & BC           & SVM, RF, XGB                                    & Atom Pairs, Topological (EState), Fragment (MACCS, Klek, PubChem), Path-based (FP4, FP4C), MD (PaDEL) & $hERG$ \\
    \cmidrule{2-5}
    Ryu (2020) \cite{Ryu2020}                 & BC           & Ensemble: DNN, GCN                              & MD (Mordred), Circular, Fragment (PubChem), Graphs & $hERG$ \\
    \cmidrule{2-5}
    Siramshetty (2020) \cite{Siramshetty2020} & BC           & RF, XGB, FFNN, LSTM                             & MD (RDKit), Circular, Embedding (AE) & $hERG$ \\
    \cmidrule{2-5}
    Wang (2020) \cite{Wang2020}               & BC           & Conv-CapsNet, RBM                               & MD, Fragment (MACCS) & $hERG$ \\
    \cmidrule{2-5}
    Creanza (2021) \cite{Creanza2021}         & BC           & LASSO-SVM                                       & Docking Scores (GLIDE, GOLD), Interactions (PLIF) & $hERG$ \\
    \cmidrule{2-5}
    Karim (2021) \cite{Karim2021}             & BC           & (DNN + GCN + Conv1D) > DNN                      & MD (Mordred), Circular, Embeddings (Smi2Vec, FP2Vec), Fragment (PubChem), Graph &$hERG$ \\
    \cmidrule{2-5}
    Lee (2021) \cite{Lee2021}                 & BC, Reg      & XGB, RF                                         & MD (RDKit), Pharmacophore & $hERG$ \\
    \cmidrule{2-5}
    Meng (2021) \cite{Meng2021}               & Reg          & SVR, RF, kNN, MLP, BRR                          & Interaction, Circular, Fragment (MACCS), Topological (EState) & $hERG$ \\
    \cmidrule{2-5}
    Moorthy (2021) \cite{Moorthy2021}         & BC           & DT, RF, SVM, NB                                 & MD (MOE), Fragment (MACCS) & $hERG$ \\
    \cmidrule{2-5}
    Sato (2021) \cite{Sato2021}               & Reg          & SVM, RF, MLP                                    & MD (RDKit, Mordred), Path-based (CDK), Circular & $hERG$ \\
    \cmidrule{2-5}
    Stergiopoulos (2021) \cite{Stergio2021}   & Reg          & Stepwise Regression                             & MD, Experimental ($\alpha-1-acid-GP$ binding, IAM permeability) & $hERG$ \\
    \cmidrule{2-5}
    Wu (2021) \cite{Wu2021}                  & BC           & MT-GAT                                          & Graphs & $hERG$ \\
    \cmidrule{2-5}
    Xiong (2021) \cite{Xiong2021}            & BC           & MT-GAT                                          & Graphs & $hERG$ \\
    \cmidrule{2-5}
    Arab (2022) \cite{Arab2022}              & MC           & SVM, RF, MLP                                    & MD (PaDEL) & $hERG$, $Na_v1.5$ \\
    \cmidrule{2-5}
    Delre (2022) \cite{Delre2022}            & BC           & RF, KNN, GBM, XGB, SVM                          & MD (DRAGON) & $hERG$ \\
    \cmidrule{2-5}
    Ding (2022) \cite{Ding2022}              & Reg          & GBM, RF, SVM, kNN                               & MD, Circular, 3D Shape-based, 4D Dynamics-based & $hERG$ \\
    \cmidrule{2-5}
    Ishihara (2022) \cite{Ishihara2022}      & BC           & XGB, NB, SVM, RF, ANN                           & MD (RDKit, Mordred) & $hERG$ \\
    \cmidrule{2-5}
    Kim (2022) \cite{Kim2022}                & BC           & D-MPNN > Attn. Pool > DNN                       & Graphs & $hERG$ \\
    \cmidrule{2-5}
    Kong (2022) \cite{Kong2022}              & BC           & LogReg, SVM, NB, MLP, RF                        & Path-based (CDK), Topological (EState), Fragment (MACCS, PubChem) & $Na_v1.5$ \\
    \cmidrule{2-5}
    Krishna (2022) \cite{Krishna2022}        & BC           & DT, NN, SVM, RF, LDA, DNN                       & MD (RDKit, OPERA) & $hERG$ \\
    \cmidrule{2-5}
    Lanevskij (2022) \cite{Lanevskij2022}    & BC, Reg      & XGB (with AFT)                                  & MD & $hERG$ \\
    \cmidrule{2-5}
    Melnikov (2022) \cite{Melnikov2022}      & Reg          & XGB                                             & MD (RDKit), Circular, Fragment (MACCS), Topological Torsion & $hERG$ \\
    \newpage
    Shan (2022) \cite{Shan2022}              & BC           & D-MPNN                                          & MD (RDKit, MOE), Circular, Fragment (MACCS, PubChem), Embedding (Mol2Vec) & $hERG$ \\
    \cmidrule{2-5}
    Zhang (2022) \cite{Zhang2022}            & BC           & FFNN, RF, XGB                                   & Circular, Fragment (MACCS) & $hERG$ \\
    \cmidrule{2-5}
    Arab (2023) \cite{Arab2023}              & BC           & (DNN + GCN) > DNN                               & MD (Mordred), Circular, Fragment (PubChem), Graphs & $hERG$, $Na_v1.5$, $Ca_v1.2$ \\
    \cmidrule{2-5}
    Banerjee (2023) \cite{Banerjee2023}      & BC, Reg      & PLS RASAR, Ridge, SVM, RF, GBM, ADA, MLP, kNN   & MD, Similarity (RASAR) & $hERG$ \\
    \cmidrule{2-5}
    Chen (2023) \cite{Chen2023}              & BC           & Graph, kNN, SVM, RF, NN                         & MD (RDKit), Circular, Atom Pairs, Fragment (MACCS), Topological Torsion & $hERG$ \\
    \cmidrule{2-5}
    Das (2023) \cite{Das2023}                & BC, Reg      & DT, RF, LogReg, AdaBoost, kNN, SVm, NB, NN, SGD & MD (PaDEL), Path-based (CDK) & $hERG$ \\
    \cmidrule{2-5}
    Feng (2023) \cite{Feng2023}              & BC           & DT, DNN                                         & 3D Shape-based, Embedding (CDDD, Transformer) & $hERG$ \\
    \cmidrule{2-5}
    Wang\sep (2023) \cite{Wang2023_CBM}      & BC           & (DNN + GAT) > DNN                               & Path-based (CDK), Circular, Fragment (PubChem), Atom Pairs, SMILES & $hERG$ \\
    \cmidrule{2-5}
    Wang\sep (2023) \cite{Wang2023_FiP}      & BC           & GCN, GSTN, SVM                                  & Circular, Graphs & $hERG$ \\
    \cmidrule{2-5}
    Vittorio (2023) \cite{Vittorio2023}      & BC           & RF                                              & MD, Circular & $hERG$ \\
    \cmidrule{2-5}
    Ylipää (2023) \cite{Ylipaa2023}          & BC           & RF, XGB, SVM, DNN, GRU-DNN, MP-GNN              & MD (Mordred), Graphs, SMILES & $hERG$ \\
    \cmidrule{2-5}
    Arab (2024) \cite{Arab2024}              & BC           & RF                                              & MD (Mordred), Circular, Fragment (PubChem) & $hERG$, $Na_v1.5$, $Ca_v1.2$ \\
    \cmidrule{2-5}
    Chen (2024) \cite{Chen2024}              & BC           & DNN                                             & Fragment (MACCS) & $hERG$, $Na_v1.5$, $Ca_v1.2$ \\
    \cmidrule{2-5}
    He (2024) \cite{He2024}                  & BC           & RoBERTa emb. > MLP                              & SMILES & $hERG$ \\
    \cmidrule{2-5}
    Liu (2024) \cite{Liu2024}                & BC           & kNN, SVM, RF, MLP, LSTM, AE                     & MD (Mold2) & $hERG$ \\
    \cmidrule{2-5}
    Sanches (2024) \cite{Sanches2024}        & BC, MC, Reg  & RF, kNN, SVM, LGBM, XGB                         & Circular & $hERG$ \\
    \cmidrule{2-5}
    Wang (2024) \cite{Wang2024}              & BC           & (GIN + DNN) > Attention > DNN                   & Circular, Fragment (MACCS, PubChem), Graphs, Embedding (Smi2Vec), Pharmacophore & $hERG$, $Na_v1.5$, $Ca_v1.2$ \\
    \cmidrule{2-5}
    Yang (2024) \cite{Yang2024}              & BC           & GNN + Attn                                      & Graphs & $hERG$ \\
    \cmidrule{2-5}
    Cano (2025) \cite{Cano2025}              & BC           & XGB Ensemble                                    & MD (MOE, alvaDesc), Topological (Kappa), Circular, Fragment (MACCS) & $hERG$ \\
    \cmidrule{2-5}
    Feng (2025) \cite{Feng2025}              & BC           & (GIN + BiGRU + MLP) > Attn                      & Circular, Fragment (PubChem, MACCS), Graphs, SMILES, Pharmacophore & $hERG$, $Na_v1.5$, $Ca_v1.2$ \\
    \cmidrule{2-5}
    Gambacorta (2025) \cite{Gambacorta2025}  & BC           & RF, SVM, XGB, kNN, ADA                          & Fragment (CSFP) & ERG, $Na_v1.5$, $Ca_v1.2$ \\
    \cmidrule{2-5}
    Han (2025) \cite{Han2025}                & BC           & 40 models from HyperOPT-sklearn                 & Circular & $hERG$ \\
    \cmidrule{2-5}
    Hossain (2025) \cite{Hossain2025}        & BC           & Logic Tensor Network                            & MD (Morgan), Path-based (CDK), Embeddings (MegaMolBART, LLaMA3.2, Gemini, QWEN1.5b) & $hERG$ \\
    \cmidrule{2-5}
    Jin (2025) \cite{Jin2025}                & BC           & ((GNN > Transformer) + MLP) > Attn > MLP        & MD, Circular, Fragment (MACCS), Graphs, Path-based (RDKit) & $hERG$ \\
    \newpage
    Jing (2025) \cite{Jing2025}              & BC           & DNN, GNN, RF, GBM, SVM, NB, kNN, MLP            & MD, Atom Types (AMBER), Circular, Fragment (MACCS), Atom Pairs, Topological Torsion  & $hERG$ \\
    \cmidrule{2-5}
    Kyro (2025) \cite{Kyro2025}              & BC, Reg, Gen & (DNN + GAT) > DNN                               & MD (RDKit), Circular, Graphs, Embeddings (BDT) & $hERG$, $Na_v1.5$, $Ca_v1.2$ \\
    \cmidrule{2-5}
    Lee (2025) \cite{Lee2025}                & BC           & (GAT + MLP) > FCNN                              & MD, Circular, Graphs & $hERG$ \\
    \cmidrule{2-5}
    Liu\sep (2025) \cite{K_Liu2025}          & BC           & NB, RF, SVM, kNN, XGB, Transformer              & MD (RDKit), Circular, Fragment (MACCS), Atom Pairs, Path-based (FP2) & $hERG$ \\
    \cmidrule{2-5}
    Liu\sep (2025) \cite{L_Liu2025}          & BC, Reg      & FCNN, Conv2D + DenseNet, EqGAT                  & Circular, Graphs, Images & $hERG$ \\
    \cmidrule{2-5}
    Mohammad (2025) \cite{Mohammad2025}      & BC           & RF, XGB, MLP, SVM, kNN, DT                      & MD (RDKit, Avalon), Path-based (CDK), Embeddings (Mol2Vec), Fragment (MACCS, Klek), Atom Pair & $hERG$ \\
    \cmidrule{2-5}
    Tran-Nguyen (2025) \cite{TranNguyen2025} & BC           & RF, XGB, DNN                                    & Interaction (PLEC) & $hERG$ \\
    \cmidrule{2-5}
    Xu (2025) \cite{Xu2025}                  & BC           & XGB                                             & Embeddings (GROVER, kBERT, GIN, Chemprop), Various (RDKit, DeepChem, MolMap, Mordred) & $hERG$ \\
    \cmidrule{2-5}
    Yang (2025) \cite{Yang2025}              & BC           & RF, SVM, GBM, LogReg                            & Circular & $hERG$ \\
    \cmidrule{2-5}
    Yu (2025) \cite{Yu2025}                  & Reg          & XGB, kNN, DNN, Ridge, Lasso                     & MD (AlvaDesc), Circular & $hERG$ \\
    \cmidrule{2-5}
    Zhang (2025) \cite{Zhang2025}            & Reg          & Ridge, Lasso, RF, XGB, kNN, GNN                 & MD (Avalon), Circular, Fragment (MACCS), Graphs & $hERG$ \\
    \cmidrule{2-5}
    Agarwal (2026) \cite{Agarwal2026}        & BC           & GCN, GraphSAGE, GIN, GAT, MPNN, AttentiveFP     & Graph & $hERG$, $Na_v1.5$, $Ca_v1.2$ \\
    \cmidrule{2-5}
    Beidja (2026) \cite{Beidja2026}          & BC           & (Transformer + DNN + LSGAT) > DNN               & SMILES, Graphs, Circular, Atom Pairs, Fragment (PubChem, Klek), Topological Torsion & $hERG$ \\
    \cmidrule{2-5}
    Chu (2026) \cite{Chu2026}                & BC           & (ViT + (GIN > GAT) + AE) > MoE Attention > DNN  & Images, Graphs, Circular, Pharmacophore & $hERG$ \\
    \cmidrule{2-5}
    Enokiya (2026) \cite{Enokiya2026}        & BC           & GINE, GCN, GraphSAGE, GATv2                     & Graphs, MD (Mordred, RDKit), FAERS signals & $hERG$ \\
    \cmidrule{2-5}
    Jovanocić (2026) \cite{Jovanovi2026}     & BC, Reg      & 3xDNN > Attn > DNN                              & Embeddings (ChemBERTa, L1000), Circular, Atom Pair, Topological Torsion, MD (RDKit, MolGpKa) & $hERG$, $Na_v1.5$, $Ca_v1.2$, $K_v7.1$ \\
    \cmidrule{2-5}
    Liang (2026) \cite{Liang2026}            & BC           & (GNN + BiGRU) > DNN                             & Graphs, Circular, Fragment (MACCS),  MD (RDKit) & $hERG$, $Na_v1.5$, $Ca_v1.2$ \\
    \cmidrule{2-5}
    Su (2026) \cite{Su2026}                  & BC           & (MPNN + DNN) > DNN                              & Fragment (MACCS, PubChem, SubFP), Pharmacophore & $hERG$ \\
    \cmidrule{2-5}
    Sun (2026) \cite{Sun2026}                & BC, Reg      & SVM                                             & Atom Types & $hERG$ \\
    \cmidrule{2-5}
    Zhang (2026) \cite{Zhang2026}            & BC           & (GNN + DNN) > DNN, kNN, RF, XGB                 & Graphs, Circular, Pharmacophore & $hERG$ \\
    \cmidrule{2-5}
    Zhao (2026) \cite{Zhao2026}              & BC           & RF, GBM, SVM, MPNN, GCN, CNN, Transformer       & MD (Mordred), Circular, Fragment (MACCS, PubChem), Graphs, SMILES, Interaction (PLIF) & $hERG$ \\
"""

In [25]:
table_s2_herg = r"""
    Ponti (2001) \cite{Ponti2001}                 & 137    & N/A         & IC50       & In paper\s\dg & Literature \\
    \cmidrule{2-6}
    Cavalli (2002) \cite{Cavalli2002}             & 37     & N/A         & IC50       & In paper      & Study \cite{Ponti2001}, FenichelDB \\
    \cmidrule{2-6}
    Roche (2002) \cite{Roche2002}                 & 472    & N/A         & IC50       & In paper\dg   & In-house Roche, known drugs \\
    \cmidrule{2-6}
    Ekins (2002) \cite{Ekins2002}                 & 22     & N/A         & IC50       & In paper      & In-house ElliLilly \\
    \cmidrule{2-6}
    Keserü (2003) \cite{Keser2003}                & 68     & N/A         & IC50       & In paper      & Study \cite{Roche2002}, FenichelDB\\
    \cmidrule{2-6}
    Pearlstein (2003) \cite{Pearlstein2003}       & 23     & N/A         & IC50       & In paper      & In-house Aventis Pharma \\
    \cmidrule{2-6}
    Aronov (2004) \cite{Aronov2004}               & 414    & 85@40\uM    & Binary     & Online        & Studies \cite{Cavalli2002, Ekins2002}, FenichelDB, Literature \\
    \cmidrule{2-6}
    Aptula (2004) \cite{Aptula2004}               & 19     & N/A         & IC50       & In paper      & Study \cite{Pearlstein2003}, Literature \\
    \cmidrule{2-6}
    Bains (2004) \cite{Bains2004}                 & 124    & N/A         & IC50       & In paper      & Literature \\
    \cmidrule{2-6}
    Fioravanzo (2004) \cite{Fioravanzo2004}       & 70     & N/A         & IC50       & In paper      & Literature \\
    \cmidrule{2-6}
    Cianchetta (2005) \cite{Cianchetta2005}       & 882    & 546@10\uM   & IC50       & SI\dg         & Studies \cite{Pearlstein2003, Cavalli2002}, In-house Sanofi \\
    \cmidrule{2-6}
    Tobita (2005) \cite{Tobita2005}               & 73     & N/A         & IC50       & In paper      & Literature \\
    \cmidrule{2-6}
    Coi (2006) \cite{Coi2006}                     & 82     & N/A         & IC50       & In paper      & Study \cite{Bains2004}, Literature \\
    \cmidrule{2-6}
    Ekins (2006) \cite{Ekins2006}                 & 134    & N/A         & IC50       & In paper      & Study \cite{Ekins2002}, Literature \\
    \cmidrule{2-6}
    Seierstad (2006) \cite{Seierstad2006}         & 439    & N/A         & IC50       & In paper\dg   & In-house J\&J \\
    \cmidrule{2-6}
    Song (2006) \cite{Song2006}                   & 90     & N/A         & IC50       & SI            & Studies \cite{Ekins2002, Cavalli2002}, In-house Locus, Literature \\
    \cmidrule{2-6}
    Sun (2006) \cite{Sun2006}                     & 1979   & N/A         & IC50       & In paper\dg   & Study \cite{Keser2003}, In-house Roche \\
    \cmidrule{2-6}
    Yoshida (2006) \cite{Yoshida2006}             & 104    & N/A         & IC50       & In paper      & Studies \cite{Ekins2002, Pearlstein2003}, FenichelDB, Literature \\
    \cmidrule{2-6}
    Du (2007) \cite{Du2007}                       & 56     & N/A         & IC50       & In paper      & Literature \\
    \cmidrule{2-6}
    Leong (2007) \cite{Leong2007}                 & 39     & N/A         & IC50       & In paper      & Studies\cite{Cavalli2002, Ekins2002, Roche2002, Pearlstein2003, Bains2004, Sun2006}, Literature \\
    \cmidrule{2-6}
    Obrezanova (2007) \cite{Obrezanova2007}       & 137    & N/A         & IC50       & SI            & Studies \cite{Pearlstein2003, Ekins2002}, FenichelDB, Literature \\
    \cmidrule{2-6}
    Gunturi (2008) \cite{Gunturi2008}             & 165    & N/A         & IC50       & In paper      & Study\cite{Yoshida2006}, Literature \\
    \cmidrule{2-6}
    Kramer (2008) \cite{Kramer2008}               & 113    & N/A         & IC50       & SI            & Studies \cite{Ekins2002, Pearlstein2003}, FenichelDB, PubChem, Literature \\
    \cmidrule{2-6}
    Li (2008) \cite{Li2008}                       & 495    & N/A         & Binary     & SI            & Studies \cite{Aronov2004, Tobita2005, Keser2003, Bains2004, Song2006, Seierstad2006, Ekins2002, Cavalli2002, Pearlstein2003, Cianchetta2005}, FenichelDB, Literature \\
    \cmidrule{2-6}
    Thai\sep (2008) \cite{Thai2008BMC}            & 313    & 197@10\uM   & IC50       & SI            & Studies \cite{Song2006, Bains2004, Pearlstein2003, Keser2003, Ekins2002, Cianchetta2005, Aptula2004, Roche2002}, FenichelDB, Literature \\
    \cmidrule{2-6}
    Thai\sep (2008) \cite{Thai2008_CBDD}          & 285    & 195<10\uM   & IC50       & SI            & Studies \cite{Song2006, Bains2004, Pearlstein2003, Keser2003, Ekins2002, Ekins2006, Cianchetta2005, Aptula2004, Roche2002, Aronov2004, Gepp2006, Dubus2006, Fioravanzo2004, Tobita2005, OBrien2005, Sun2006, Li2008, Thai2008BMC}, FenichelDB, Literature \\
    \cmidrule{2-6}
    Coi (2009) \cite{Coi2009}                     & 155    & N/A         & IC50       & In paper      & Studies\cite{Ekins2002, Pearlstein2003}, Literature \\
    \newpage
    Nisius\sep (2009) \cite{Nisius2009CBDD}       & 242    & 138@10\uM   & IC50       & SI            & Studies \cite{Ekins2006, Yoshida2006, Song2006, Keser2003, Cavalli2002, Bains2004, Thai2008BMC}, Literature \\
    \cmidrule{2-6}
    Nisius\sep (2009) \cite{Nisius2009JCIM}       & 275    & 160<10\uM   & IC50       & SI            & Studies \cite{Kramer2008, Ekins2006, Yoshida2006, Song2006, Keser2003, Cavalli2002, Thai2008BMC, Bains2004}, Literature \\
    \cmidrule{2-6}
    Polak (2009) \cite{Polak2009}                 & 201    & N/A         & IC50       & In paper      & Study \cite{Roche2002}, FenichelDB,  QTDrugs \\
    \cmidrule{2-6}
    Thai (2009) \cite{Thai2009}                   & 307    & N/A         & Mixed      & SI            & Studies\cite{Song2006, Bains2004, Pearlstein2003, Keser2003, Ekins2002, Cianchetta2005, Aptula2004, Roche2002},  FenichelDB, Literature \\
    \cmidrule{2-6}
    Doddareddy (2010) \cite{Doddareddy2010}       & 2644   & 1112<10\uM  & Binary     & SI            & Studies \cite{Li2008, Obrezanova2007, Coi2006, Tobita2005}, Literature \\
    \cmidrule{2-6}
    Obrezanova (2010) \cite{Obrezanova2010}       & 168    & 117@10\uM   & Binary     & SI            & Study \cite{Obrezanova2007}, Literature \\
    \cmidrule{2-6}
    Su (2010) \cite{Su2010}                       & 250    & N/A         & IC50       & SI            & Studies\cite{Li2008, Song2006, Yoshida2006, Thai2008BMC} \\
    \cmidrule{2-6}
    Cuny (2011) \cite{Cuny2011}                   & 529    & 178         & IC50       & In paper\dg   & Studies \cite{Ponti2001, Yoshida2006, Song2006, Keser2003, Sun2006, Pearlstein2003, Ekins2006}, DrugBank, Literature \\
    \cmidrule{2-6}
    Obiol-Pardo (2011) \cite{ObiolPardo2011}      & 355    & N/A         & IC50       & SI            & Literature \\
    \cmidrule{2-6}
    Robinson (2011) \cite{Robinson2011}           & 220    & N/A         & IC50       & SI            & Literature \\
    \cmidrule{2-6}
    Sinha (2011) \cite{Sinha2011}                 & 157    & N/A         & IC50       & In paper      & Studies \cite{Coi2006, Seierstad2006, Ekins2002, Pearlstein2003, Fioravanzo2004, Song2006, Li2008, Cianchetta2005} \\
    \cmidrule{2-6}
    Broccatelli (2012) \cite{Broccatelli2012}     & 1051   & 185<1\uM    & IC50       & SI            & ChEMBL, Tox-Portal \\
    \cmidrule{2-6}
    Polak (2012) \cite{Polak2012}                 & 98     & N/A         & IC50       & In paper      & Literature \\
    \cmidrule{2-6}
    Tan (2012) \cite{Tan2012}                     & 113    & N/A         & IC50       & In paper      & Literature \\
    \cmidrule{2-6}
    Wang (2012) \cite{Wang2012}                   & 806    & N/A         & IC50       & From \cite{Wang2016} & Studies \cite{Li2008, Polak2009, Thai2008BMC, Song2006, Tobita2005, Du2007, Roche2002, Obrezanova2007, Obrezanova2010, Nisius2009CBDD}, WOMBAT, Literature \\
    \cmidrule{2-6}
    Wiśniowska (2012) \cite{Winiowska2012}        & 123    & N/A         & IC50       & In paper      & Literature \\
    \cmidrule{2-6}
    Czodrowski (2013) \cite{Czodrowski2013}       & 4415   & 3047@10\uM  & Binary\ddg & SI            & ChEMBL \\
    \cmidrule{2-6}
    Braga (2015) \cite{Braga2015}                 & 5984   & 2565<10\uM  & Binary\ddg & \href{https://github.com/LabMolUFG/Pred_hERG}{GitHub}\p & ChEMBL \\
    \cmidrule{2-6}
    Du (2015) \cite{Du2015}                       & 306895 & N/A         & \%Inh\ddg  & SI, \href{https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/7BVDG8}{Online} & MLSMR collection \\
    \cmidrule{2-6}
    Chavan (2016) \cite{Chavan2016}               & 172    & 93@10\uM    & IC50       & SI & FenichelDB, OCHEM \\
    \cmidrule{2-6}
    Didziapetris (2016) \cite{Didzia2016}         & 6690   & 3908@10\uM  & IC50       & SI & Studies \cite{Polak2009, Doddareddy2010, Broccatelli2012}, ChEMBL, Literature \\
    \cmidrule{2-6}
    Wang (2016) \cite{Wang2016}                   & 806    & N/A         & IC50       & SI & Study \cite{Wang2012} \\
    \cmidrule{2-6}
    Zhang (2016) \cite{Zhang2016}                 & 1570   & N/A         & IC50       & SI & Studies \cite{Doddareddy2010, Wang2012}, ChEMBL, hERGCentral \\
    \cmidrule{2-6}
    Chemi (2017) \cite{Chemi2017}                 & 421    & 22<50nM     & IC50       & SI & Literature \\
    \cmidrule{2-6}
    Sun (2017) \cite{Sun2017}                     & 3024   & 1008@30\uM  & IC50       & SI & qHTS Thalium Flux assay \\
    \cmidrule{2-6}
    Munawar (2018) \cite{Munawar2018}             & 207    & N/A         & IC50       & SI & Studies \cite{Kramer2008, Polak2009}, ChEMBL, FenichelDB\\
    \cmidrule{2-6}
    Sato (2018) \cite{Sato2018}                   & 291219 & 9890@10\uM  & Binary     & \href{https://github.com/AI-amateur/DMPNN-hERG}{From} \cite{Shan2022, Feng2023, Cano2025}\dg & Databases \cite{ChEMBL,GOSTAR,PubChem,hERGCentral} \\
    \cmidrule{2-6}
    Siramshetty (2018) \cite{Siramshetty2018}     & 5804   & 4096@10\uM  & Binary\ddg & \href{https://github.com/AGPreissner/Publications}{GitHub} & Studies\cite{Sun2017, Doddareddy2010, Li2008, Robinson2011}, ChEMBL\\
    \cmidrule{2-6}
    Cai (2019) \cite{Cai2019}                     & 7889   & 4355@10\uM  & Binary\ddg & SI & Studies\cite{Chemi2017, Doddareddy2010, Wang2016, Didzia2016}, ChEMBL\\
    \cmidrule{2-6}
    Konda (2019) \cite{Konda2019}                 & 8705   & 5286@10\uM  & IC50       & SI & ChEMBL\\
    \cmidrule{2-6}
    Zhang(2019) \cite{Zhang2019}                  & 697    & 211@5\uM    & Binary     & SI & Studies\cite{Li2008, Chavan2016}, PubChem  \\
    \cmidrule{2-6}
    Choi (2020) \cite{Choi2020}                   & 5299   & 3735<1\uM   & IC50       & SI & ChEMBL, Literature \\
    %\cmidrule{2-6}
    \newpage
    Ryu (2020) \cite{Ryu2020}                     & 14440  & 6632@10\uM  & N/A        & From \cite{Khalifa2020}\dg & Studies\cite{Cai2019, Didzia2016, Doddareddy2010, Munawar2018, Thai2008BMC}, ChEMBL, BindingDB, In-house \\
    \cmidrule{2-6}
    Siramshetty (2020) \cite{Siramshetty2020}     & 8984   & 2230        & Binary     & \href{https://github.com/ncats/herg-ml}{GitHub} & ChEMBL, PubChem, In-house NCATS \\
    \cmidrule{2-6}
    Creanza (2021) \cite{Creanza2021}             & 8337   & 1308<1\uM   & IC50       & SI & ChEMBL\\
    \cmidrule{2-6}
    \dr{Lee (2021) \cite{Lee2021}}                & 3448   & \dr{N/A}    & IC50       & \dr{\href{https://github.com/NIDA-IRP-CCMB/QSAR_DAT-hERG}{GitHub}} & \dr{ChEMBL} \\
                                                  & 678    &             & Ki         &                                                                    & \\
    \cmidrule{2-6}
    Meng(2021) \cite{Meng2021}                    & 9215   & N/A         & IC50       & SI & ChEMBL\\
    \cmidrule{2-6}
    Moorthy (2021) \cite{Moorthy2021}             & 136*   & N/A         & IC50       & SI & Literature \\
    \cmidrule{2-6}
    Stergiopoulos (2021) \cite{Stergio2021}       & 90     & N/A         & IC50       & In paper & Study\cite{Sato2018} \\
    \cmidrule{2-6}
    Wu (2021) \cite{Wu2021}                       & 1565   & 955@10\uM   & Binary\ddg & \href{https://github.com/wzxxxx/MGA}{GitHub}  & Literature \\
    \cmidrule{2-6}
    Arab (2022) \cite{Arab2022}                   & 8930   & N/A         & IC50       & \href{https://github.com/issararab/ToxTree}{GitHub} & Study\cite{Konda2019}, ChEMBL, PubChem \\
    \cmidrule{2-6}
    Delre (2022) \cite{Delre2022}                 & 8755   & N/A         & IC50       & \href{https://github.com/PDelre93/hERG-QSAR}{GitHub} & ChEMBL\\
    \cmidrule{2-6}
    Ding (2022) \cite{Ding2022}                   & 7836   & N/A         & IC50       & SI & Studies\cite{Munawar2019, Didzia2016}, ChEMBL\\
    \cmidrule{2-6}
    \dr{Kim (2022) \cite{Kim2022}}                & 304045 & N/A         & IC50       & \dr{\href{https://github.com/GIST-CSBL/BayeshERG}{GitHub}} & Study\cite{Du2015} \\
                                                  & 14322  & 8488@10\uM  & Binary     &                                                            & Study\cite{Cai2019}, Databases\cite{ChEMBL,PubChem,BindingDB} \\
    \cmidrule{2-6}
    Krishna (2022) \cite{Krishna2022}             & 8311   & 1970@1\uM   & IC50       & SI & ChEMBL, Tox21 \cite{Tox21} \\
    \cmidrule{2-6}
    Lanevskij (2022) \cite{Lanevskij2022}         & 9311   & 4862@10\uM  & IC50       & SI & Study\cite{Didzia2016}, Literature \\
    \cmidrule{2-6}
    Zhang (2022) \cite{Zhang2022}                 & 12850  & 6907@10\uM  & Binary     & \href{https://figshare.com/articles/dataset/Untitled_Item/19388249}{Online} & Study\cite{Sun2017}, ChEMBL, Literature \\
    \cmidrule{2-6}
    Arab (2023) \cite{Arab2023}                   & 22246  & N/A         & IC50       & \href{https://zenodo.org/records/8359714}{Zenodo} & Studies\cite{Didzia2016,Konda2019,Doddareddy2010,Munawar2019}, Databases \cite{ChEMBL, PubChem, BindingDB, hERGCentral}, U.S. Patents \\
    \cmidrule{2-6}
    Banerjee (2023) \cite{Banerjee2023}           & 261    & N/A         & IC50       & SI & Studies\cite{Kar2012, Stergio2021, Ekins2006} \\
    \cmidrule{2-6}
    Chen (2023) \cite{Chen2023}                   & 4556   & N/A         & IC50       & SI & Studies\cite{Doddareddy2010, Zhang2016}, ChEMBL\\
    \cmidrule{2-6}
    Das (2023) \cite{Das2023}                     & 6766   & 4000@10\uM  & IC50       & SI & Studies\cite{Thai2008BMC, Jia2008, Tobita2005, Ermondi2009, Sinha2011, Cavalli2002} \\
    \cmidrule{2-6}
    Wang (2023) \cite{Wang2023_CBM}               & 10355  & 4479@10\uM  & Binary     & \href{https://github.com/zhaoqi106/DMFGAM}{GitHub} & Studies\cite{Konda2019, Liu2020, Munawar2019, Negami2019}, ChEMBL\\
    \cmidrule{2-6}
    Vittorio (2023) \cite{Vittorio2023}           & 12789  & N/A         & pK         & \href{https://zenodo.org/records/7551782}{Zenodo} & ChEMBL, GOSTAR \\
    \cmidrule{2-6}
    \tr{Sanches (2024) \cite{Sanches2024}}        & 7307   & 3731@10\uM  & Binary     & \tr{SI} & \tr{ChEMBL} \\
                                                  & 6142   & \dr{N/A}    & 4-class    & & \\
                                                  & 4315   &             & IC50       & & \\
    \cmidrule{2-6}
    Yang (2024) \cite{Yang2024}                   & 14322  & 8488@10\uM  & Binary     & \href{https://github.com/Tianbiao-Yang/AttenhERG}{GitHub} & Study\cite{Kim2022}, ChEMBL, PubChem \\
    \cmidrule{2-6}
    Gambacorta (2025) \cite{Gambacorta2025}       & 10035  & 5042@10\uM  & IC50       & \href{https://github.com/f48r1/cupid}{GitHub} & ChEMBL \\
    \cmidrule{2-6}
    Han (2025) \cite{Han2025}                     & 6744   & 3465@10\uM  & Binary     & \href{https://github.com/CADD-SC/ADMET_Prediction_Models}{GitHub} & ChEMBL\\
    \cmidrule{2-6}
    Hossain (2025) \cite{Hossain2025}             & 20409  & 9826@10\uM  & Binary     & \href{https://github.com/hossain013/hERG-LTN}{GitHub} & Studies\cite{Karim2021, Wang2016}, Databases \cite{ChEMBL,BindingDB,PubChem, GTP}\\
    \cmidrule{2-6}
    Lee (2025) \cite{Lee2025}                     & 23381  & 14183@10\uM & Binary     & \href{https://github.com/bmil-jnu/hERGAT}{GitHub} & Studies\cite{Li2008, Wang2016, Zhang2019, Kim2022}, ChEMBL, PubChem \\
    \cmidrule{2-6}
    \dr{Tran-Nguyen (2025) \cite{TranNguyen2025}} & 299927 & 1937@20\uM  & Binary     & \dr{\href{https://github.com/vktrannguyen/HERGAI}{GitHub}} & \dr{ChEMBL, PubChem} \\
                                                  & 2340   & N/A         & IC50       & & \\
    \cmidrule{2-6}
    Xu (2025) \cite{Xu2025}                       & 116    & 53@10\uM    & Mixed      & SI & Literature \\
    %\cmidrule{2-6}
    \newpage
    \dr{Yu (2025) \cite{Yu2025}}                  & 10798  & \dr{N/A}    & \dr{IC50}  & \dr{\href{http://ssbio.cau.ac.kr/software/hergboost}{Platform}} & Studies\cite{Didzia2016,Li2008,Liu2020,Munawar2019,Negami2019,Wang2016}, ChEMBL, BindingDB \\
                                                  & 796    &             &            &                                                                 & Studies\cite{Lanevskij2022,Ryu2020,Karim2021} \\
    \cmidrule{2-6}
    Zhang (2025) \cite{Zhang2025}                 & 13890  & N/A         & IC50       & \href{https://github.com/ZYX2222/hERG_binding_regressor}{GitHub} & Study\cite{Krishna2022}, ChEMBL\\
    \cmidrule{2-6}
    Agarwal (2026) \cite{Agarwal2026}             & 34124  & 14444@10\uM & Binary     & \href{https://github.com/pip700/cardiotox_prediction}{GitHub} & Studies\cite{Gambacorta2025, Arab2024}, PubChem \\
    \cmidrule{2-6}
    Jovanović (2026) \cite{Jovanovi2026}          & 331127 & 11881@10\uM & IC50       & \href{https://github.com/AppliedScientific/CardioSafe-benchmark}{GitHub} & ChEMBL \\
    \cmidrule{2-6}
    Sun (2026) \cite{Sun2026}                     & 1333   & N/A         & IC50       & SI & ChEMBL \\
    \cmidrule{2-6}
    Zhang (2026) \cite{Zhang2026}                 & 16543  & 6333        & Binary     & SI & Studies\cite{Krishna2022, Cai2019}, ChEMBL, Tox21, Literature \\
    \cmidrule{2-6}
    Zhao (2026) \cite{Zhao2026}                   & 15806  & 5965@1\uM   & Binary     & \href{https://github.com/ConfusedAnt/FEAOF}{GitHub} & ChEMBL\\
"""

table_s2_nav1_5 = r"""
    \dr{Khalifa (2020) \cite{Khalifa2020}}  & 2596  & \dr{N/A}  & IC50       & \dr{SI} & \dr{ChEMBL, BindingDB, In-house Jubilant BioSys} \\
                                            & 714   &           & Inhibition &         & \\
    \cmidrule{2-6}
    Arab (2022) \cite{Arab2022}             & 1767  & N/A       & IC50       & \href{https://github.com/issararab/ToxTree}{GitHub} & ChEMBL, PubChem \\
    \cmidrule{2-6}
    Arab (2023) \cite{Arab2023}             & 2069  & N/A       & IC50       & \href{https://zenodo.org/records/8359714}{Zenodo} & Databases \cite{ChEMBL, PubChem, BindingDB}, U.S. Patents \\
    \cmidrule{2-6}
    Gambacorta (2025) \cite{Gambacorta2025} & 1436 & 679@10\uM  & IC50       & \href{https://github.com/f48r1/cupid}{GitHub} & ChEMBL \\
    \cmidrule{2-6}
    Agarwal (2026) \cite{Agarwal2026}       & 3217 & 1894@10\uM & Binary     & \href{https://github.com/pip700/cardiotox_prediction}{GitHub} & Studies\cite{Gambacorta2025, Arab2024}, PubChem \\
    \cmidrule{2-6}
    Jovanović (2026) \cite{Jovanovi2026}    & 3160 & 1240@10\uM & IC50       & \href{https://github.com/AppliedScientific/CardioSafe-benchmark}{GitHub} & ChEMBL \\
"""

table_s2_cav1_2 = r"""
    Arab (2023) \cite{Arab2023}             & 802  & N/A       & IC50   & \href{https://zenodo.org/records/8359714}{Zenodo} & Databases \cite{ChEMBL, PubChem, BindingDB}, U.S. Patents \\
    \cmidrule{2-6}
    Gambacorta (2025) \cite{Gambacorta2025} & 734  & 405@10\uM & IC50   & \href{https://github.com/f48r1/cupid}{GitHub} & ChEMBL \\
    \cmidrule{2-6}
    Agarwal (2026) \cite{Agarwal2026}       & 1564 & 958@10\uM & Binary & \href{https://github.com/pip700/cardiotox_prediction}{GitHub} & Studies\cite{Gambacorta2025, Arab2024}, PubChem \\
    \cmidrule{2-6}
    Jovanović (2026) \cite{Jovanovi2026}    & 1138 & 548@10\uM & IC50   & \href{https://github.com/AppliedScientific/CardioSafe-benchmark}{GitHub} & ChEMBL \\
"""

table_s2_kv7_1 = r"""
    Obiol-Pardo (2011) \cite{ObiolPardo2011} & 162 & N/A      & IC50  & SI & Literature \\
    \cmidrule{2-6}
    Jovanović (2026) \cite{Jovanovi2026}     & 115 & 30@10\uM & IC50  & \href{https://github.com/AppliedScientific/CardioSafe-benchmark/tree/main/data}{GitHub} & ChEMBL \\
"""

In [26]:
table_s3_herg = r"""
    \dr{Roche (2002) \cite{Roche2002}}             & Random CV                           & \dr{DL (MLP)}         & \dr{<1,>10}     & -       & -       & 0.70    & 0.89    & -       & 0.61    \\
                                                   & Ext                                 &                       &                 & -       & -       & 0.71    & 0.93    & -       & 0.66    \\
    \cmidrule{2-10}
	\dr{Aronov (2004) \cite{Aronov2004}}           & Random CV                           & \dr{PH}               & \dr{-}          & 0.82    & -       & 0.71    & 0.85    & -       & -       \\
                                                   & Ext                                 &                       &                 & -       & -       & 0.63    & 0.71    & -       & -       \\
	\cmidrule{2-10}
    Bains (2004) \cite{Bains2004}                  & Random                              & ML (DT)               & 1               & 0.89\s  & -       & 1.00    & 0.83    & -       & -       \\
    \cmidrule{2-10}
    Fioravanzo (2004) \cite{Fioravanzo2004}        & D-Optimal                           & ST (PLS)              & 5               & -       & -       & 0.96    & 0.67    & -       & -       \\
    \cmidrule{2-10}
    O'Brien (2005) \cite{OBrien2005}               & Random                              & ML (Ensemble)         & -               & 0.91    & -       & 0.87    & -       & -       & -       \\
    \cmidrule{2-10}
    \dr{Tobita (2005) \cite{Tobita2005}}           & \dr{Random CV}                      & \dr{ML (SVM)}         & ~40             & 0.95    & -       & 0.97    & 0.87    & -       & -       \\
                                                   &                                     &                       & 1               & 0.90    & -       & 0.86    & 0.93    & -       & -       \\
    \cmidrule{2-10}
    Dubus (2006) \cite{Dubus2006}                  & Random CV                           & ML (DT)               & <1,>10          & 0.96    & -       & 0.98    & 0.89    & -       & -       \\
    \cmidrule{2-10}
    Gepp (2006) \cite{Gepp2006}                    & -                                   & ML (DT)               & -               & 0.80    & -       & 0.93    & 0.83    & -       & -       \\
    \cmidrule{2-10}
    Sun (2006) \cite{Sun2006}                      & Ext\cite{Keser2003}                 & ML (NB)               & 30              & 0.88    & -       & -       & -       & -       & -       \\
    \cmidrule{2-10}
    Chekmarev (2008) \cite{Chekmarev2008}          & Random CV                           & ML (SVM)              & <1,>10          & 0.78    & -       & 0.73    & 0.74    & -       & -       \\
    \cmidrule{2-10}
    \dr{Jia (2008) \cite{Jia2008}}                 & Random CV                           & \dr{ML (SVM)}         & \dr{30}         & 0.85    & -       & 0.84    & 0.87    & 0.77    & -       \\
                                                   & Ext\cite{Keser2003}                 &                       &                 & 0.94    & -       & -       & -       & -       & -       \\
    \cmidrule{2-10}
    \dr{Li (2008) \cite{Li2008}}                   & LOO CV                              & \dr{ML (SVM)}         & \dr{40}         & -       & -       & -       & -       & -       & 0.40    \\
                                                   & Ext \cite{WOMBAT}                   &                       &                 & 0.72    & -       & -       & -       & -       & -       \\
    \cmidrule{2-10}
    Thai\sep (2008) \cite{Thai2008BMC}             & Random CV                           & ML (NB)               & <1,>10          & 0.93    & -       & 0.90    & 0.95    & -       & -       \\
    \cmidrule{2-10}
    \dr{Thai\sep (2008) \cite{Thai2008_CBDD}}      & Random                              & \dr{DL (CP-NN)}       & \dr{<1,<10,>10} & 0.82    & -       & -       & -       & -       & -       \\
                                                   & Diverse train                       &                       &                 & 0.85    & -       & -       & -       & -       & -       \\
    \cmidrule{2-10}
    \dr{Wang (2008) \cite{Wang2008}}               & Random CV                           & ML (SVM)              & \dr{<1,<10,>10} & 0.95    & -       & 0.92    & 0.98    & -       & -       \\
                                                   & Ext                                 & ML (LogReg)           &                 & 0.78    & -       & 0.91    & 0.63    & -       & -       \\
    \cmidrule{2-10}
    Nisius\sep (2009) \cite{Nisius2009CBDD}        & Random CV                           & ML (SVM)              & 10              & 0.85    & -       & 0.87    & 0.84    & -       & -       \\
    \cmidrule{2-10}
    Nisius\sep (2009) \cite{Nisius2009JCIM}        & Distance                            & ML (kNN)              & <1,<10,>10      & 0.90    & -       & 0.83    & 0.93    & -       & -       \\
    \cmidrule{2-10}
    \dr{Thai (2009) \cite{Thai2009}}               & Random CV                           & \dr{DL (CP-NN)}       & \dr{<1,<10,>10} & 0.68    & -       & 0.70    & 0.57    & -       & -       \\
                                                   & Diverse                             &                       &                 & 0.72    & -       & 0.70    & 0.64    & -       & -       \\
    \cmidrule{2-10}
    Doddareddy (2010) \cite{Doddareddy2010}        & Random CV                           & ML (SVM)              & <10,>30         & 0.88    & -       & 0.83    & 0.92    & -       & -       \\
    \cmidrule{2-10}
    Obrezanova (2010) \cite{Obrezanova2010}        & Distance                            & ML (GP)               & 10              & 0.86    & -       & 0.97    & 0.65    & -       & -       \\
    \newpage

    \dr{Su (2010) \cite{Su2010}}                   & Random                              & \dr{ML (MLR)}         & \dr{40}         & 0.84    & -       & 0.94    & 0.37    & -       & -       \\
                                                   & Ext\cite{PubChem}                   &                       &                 & 0.83    & -       & 0.85    & 0.84    & -       & -       \\
    \cmidrule{2-10}
    \dr{Wiśniowska (2010) \cite{Wisniowska2010}}   & Random CV                           & \dr{ML (RF)}          & \dr{10}         & 0.85    & -       & 0.88    & 0.92    & 0.92    & -       \\
                                                   & Ext                                 &                       &                 & 0.73    & -       & 0.62    & 0.81    & -       & -       \\
    \cmidrule{2-10}
    Cuny (2011) \cite{Cuny2011}                    & Distance                            & ML (kNN)              & MT              & 0.82    & -       & 0.83    & 0.80    & -       & -       \\
    \cmidrule{2-10}
    \dr{Kim (2011) \cite{Kim2011}}                 & Random                              & \dr{ML (NB)}          & \dr{10}         & 0.95    & -       & 0.82    & 1.00    & >0.90   & -       \\
                                                   & In-house                            &                       &                 & 0.86    & -       & 0.80    & 0.92    & -       & -       \\
    \cmidrule{2-10}
    \dr{Robinson (2011) \cite{Robinson2011}}       & Random CV                           & \dr{ML (Winnow)}      & \dr{1}          & -       & -       & -       & -       & -       & 0.87    \\
                                                   & Ext                                 &                       &                 & -       & -       & -       & -       & -       & 0.40    \\
    \cmidrule{2-10}
    \dr{Shen (2011) \cite{Shen2011}}               & Random CV                           & \dr{ML (SVM)}         & \dr{10}         & 0.95    & -       & 0.90    & 0.96    & -       & -       \\
                                                   & Ext                                 &                       &                 & 0.87    & -       & -       & -       & -       & -       \\
    \cmidrule{2-10}
    \dr{Broccatelli (2012) \cite{Broccatelli2012}} & Random                              & \dr{ML (Ensemble)}    & \dr{<1,>16.6}   & 0.90    & -       & 0.93    & 0.89    & -       & -       \\
                                                   & In-house                            &                       &                 & 0.85    & -       & 0.64    & 1.00    & -       & -       \\
    \cmidrule{2-10}
    Kar (2012) \cite{Kar2012}                      & Random                              & ST (LDA)              & 5.62            & 0.80    & -       & 0.89    & 0.71    & 0.85    & -       \\
    \cmidrule{2-10}
    Tan (2012) \cite{Tan2012}                      & Diverse                             & ML (Ensemble)         & 1               & 0.83    & -       & 0.86    & 0.76    & -       & -       \\
    \cmidrule{2-10}
    Wang (2012) \cite{Wang2012}                    & Random                              & ML (NB)               & 5MT             & 0.85    & -       & -       & -       & -       & -       \\
    \cmidrule{2-10}
    \dr{Czodrowski (2013) \cite{Czodrowski2013}}   & \dr{Random CV}                      & \dr{ML (RF)}          & 1               & 0.97    & -       & 0.14    & -       & 0.56    & 0.24    \\
                                                   &                                     &                       & 10              & 0.69    & -       & 0.53    & -       & 0.67    & 0.35    \\
    \cmidrule{2-10}
    Kireeva (2013) \cite{Kireeva2013}              & Random Nested CV                    & ML (SVM)              & 10              & -       & 0.81    & -       & -       & -       & -       \\
    \cmidrule{2-10}
    Braga (2014) \cite{Braga2014}                  & Random CV                           & ML (Ensemble)         & 10MT            & 0.90    & 0.91    & 0.89    & 0.93    & 0.91    & 0.76    \\
    \cmidrule{2-10}
    Kratz (2014) \cite{Kratz2014}                  & N/A                                 & PH                    & N/A             & -       & 0.85    & -       & -       & 0.91    & -       \\
    \cmidrule{2-10}
    \dr{Liu (2014) \cite{Liu2014}}                 & Random CV                           & \dr{ML (NB)}          & \dr{<10,>30}    & 0.91    & -       & 0.90    & 0.92    & -       & -       \\
                                                   & Ext\cite{Doddareddy2010}            &                       &                 & 0.58    & -       & 0.61    & 0.57    & -       & -       \\
    \cmidrule{2-10}
    \dr{Braga (2015) \cite{Braga2015}}             & \dr{Random CV}                      & \dr{ML (SVM)}         & 10              & 0.83\dg & -       & 0.89\dg & 0.79\dg & -       & -       \\
                                                   &                                     &                       & <1,<10,>10      & 0.83    & -       & 0.77    & -       & -       & -       \\
    \cmidrule{2-10}
    \dr{Chavan (2016) \cite{Chavan2016}}           & Random CV                           & \dr{ML (Ensemble)}    & \dr{5}          & 0.70    & -       & 0.78    & 0.61    & -       & -       \\
                                                   & Ext \cite{PubChem}                  &                       &                 & 0.55    & -       & 0.63    & 0.54    & -       & -       \\
    \cmidrule{2-10}
    \dr{Didziapetris (2016) \cite{Didzia2016}}     & Random                              & \dr{ML (GBM)}         & \dr{10}         & 0.74    & -       & 0.84    & 0.62    & 0.72    & 0.46    \\
                                                   & Ext (Temporal)                      &                       &                 & 0.78    & -       & 0.75    & 0.82    & 0.83    & 0.68    \\
    \cmidrule{2-10}
    Wang (2016) \cite{Wang2016}                    & Diverse train                       & ML (SVM)              & 40              & 0.82    & -       & 0.91    & 0.65    & 0.84    & 0.59    \\
    \cmidrule{2-10}
    Zhang (2016) \cite{Zhang2016}                  & Random CV                           & ML (SVM)              & 30MT            & 0.84    & -       & 0.96    & 0.38    & 0.83    & -       \\
    \cmidrule{2-10}
    \dr{Li (2017) \cite{Li2017}}                   & Random CV                           & \dr{ML (Ensemble)}    & \dr{1}          & 0.84    & -       & 0.92    & 0.63    & 0.89    & 0.57    \\
                                                   & Ext \cite{Doddareddy2010}           &                       &                 & 0.90    & -       & 0.96    & 0.66    & 0.91    & 0.67    \\
    \cmidrule{2-10}
    \dr{Sun (2017) \cite{Sun2017}}                 & Random CV                           & \dr{ML (SVM)}         & \dr{30}         & -       & -       & 0.86    & 0.89    & 0.93    & -       \\
                                                   & Ext\cite{Keser2003}                 &                       &                 & -       & -       & -       & -       & 0.86    & -       \\
    \cmidrule{2-10}
    Alves (2018) \cite{Alves2018}                  & Random CV                           & ML (Ensemble)         & 10              & 0.80      & 0.80    & 0.87    & 0.72    & -       & -       \\
    %\cmidrule{2-10}
    \dr{Siramshetty (2018) \cite{Siramshetty2018}} & Stratified Random CV                & ML (RF)               & <1,>10          & -       & 0.87\dg & 0.82\dg & 0.92\dg & 0.94    & -       \\
                                                   & Ext \cite{Doddareddy2010}           & ML (SVM)              & 10              & -       & -       & -       & -       & 0.89\dg & -       \\
    \cmidrule{2-10}
    Wacker (2018) \cite{Wacker2018}                & Ext \cite{CiPA_promiscuity}         & ML (XGB)              & 10              & -       & -       & -       & -       & 0.93    & -       \\
    \cmidrule{2-10}
    Cai (2019) \cite{Cai2019}                      & Diverse train                       & DL (DNN)              & <10,>MT         & 0.93    & -       & -       & -       & 0.97    & -       \\
    \cmidrule{2-10}
    \dr{Konda (2019) \cite{Konda2019}}             & Random CV                           & ML (RF)               & 10              & 0.81    & -       & 0.70    & 0.88    & -       & -       \\
                                                   & External                            & ML (Ensemble)         & 30              & 0.92    & -       & 0.96    & 0.79    & -       & 0.79    \\
    \cmidrule{2-10}
    \dr{Lee (2019) \cite{Lee2019}}                 & Random CV                           & \dr{DL (NN)}          & \dr{10}         & 0.90    & -       & 0.97    & 0.32    & 0.76    & 0.37    \\
                                                   & Ext                                 &                       &                 & 0.80    & -       & 0.60    & 1.00    & -       & 0.66    \\
    \cmidrule{2-10}
    Ogura (2019) \cite{Ogura2019}                  & Random                              & ML (SVM)              & 10              & 0.98    & 0.83    & 0.67    & 0.99    & 0.96    & -       \\
    \cmidrule{2-10}
    \dr{Zhang (2019) \cite{Zhang2019}}             & Random                              & \dr{DL (NN}           & 5               & 0.78    & -       & 0.89    & 0.71    & -       & -       \\
                                                   & Ext \cite{PubChem}                  &                       & 20\% Inh@10uM   & 0.82    & -       & 0.46    & 0.77    & -       & -       \\
    \cmidrule{2-10}
    \dr{Choi (2020) \cite{Choi2020}}               & Random                              & \dr{ML (RF)}          & \dr{<1,>10}     & 0.90    & -       & -       & -       & 0.95    & -       \\
                                                   & Ext                                 &                       &                 & 0.81    & -       & -       & -       & -       & -       \\
    \cmidrule{2-10}
    Kim (2020) \cite{Kim2020}                      & Ext \cite{Didzia2016}               & DL (DNN)              & 10              & -       & -       & -       & -       & 0.93    & -       \\
    \cmidrule{2-10}
    \dr{Liu (2020) \cite{Liu2020}}                 & Random CV                           & \dr{ML (Ensemble)}    & 30              & 0.85    & -       & 0.95    & 0.52    & 0.89    & -       \\
                                                   & Ext \cite{hERGCentral}              &                       & 1,10            & 0.76    & -       & 0.76    & 0.64    & 0.79    & -       \\
    \cmidrule{2-10}
    \dr{Ryu (2020) \cite{Ryu2020}}                 & Stratified Random                   & \dr{DL (GCN, DNN)}    & \dr{10}         & 0.81    & -       & 0.93    & 0.69    & -       & 0.64    \\
                                                   & Ext \cite{Munawar2018, Thai2008BMC} &                       &                 & 0.77    & -       & 0.83    & 0.64    & -       & 0.48    \\
    \cmidrule{2-10}
    \dr{Siramshetty (2020) \cite{Siramshetty2020}} & Random CV                           & \dr{ML (RF)}          & \dr{<1,>10}     & -       & 0.84    & 0.73    & 0.95    & -       & -       \\
                                                   & Scaffold CV                         &                       &                 & -       & 0.80    & 0.66    & 0.94    & 0.90    & -       \\
    \cmidrule{2-10}
    \dr{Wang (2020) \cite{Wang2020}}               & Random CV \cite{Doddareddy2010}     & \dr{DL (CapsNet)}     & \dr{<10,>30}    & 0.92    & -       & 0.92    & 0.93    & 0.94    & 0.84    \\
                                                   & Ext \cite{Doddareddy2010}           &                       &                 & 0.79    & -       & 0.94    & 0.71    & 0.81    & 0.60    \\
    \cmidrule{2-10}
    Creanza (2021) \cite{Creanza2021}              & Random CV                           & ML (SVM)              & <10,>80         & 0.79    & -       & -       & -       & 0.86    & -       \\
    \cmidrule{2-10}
    \tr{Karim (2021) \cite{Karim2021}}             & Random CV                           & \tr{DL (MM-DNN)}      & \tr{10}         & 0.86    & -       & 0.86    & 0.86    & 0.93    & 0.72    \\
                                                   & Ext \cite{Ryu2020}                  &                       &                 & 0.81    & -       & 0.79    & 0.83    & -       & 0.60    \\
                                                   & Ext \cite{Siramshetty2020}          &                       &                 & 0.75    & -       & 0.70    & 0.79    & -       & 0.22    \\
    \cmidrule{2-10}
    \dr{Lee (2021) \cite{Lee2021}}                 & Random CV: binding                  & \dr{ML (XGB)}         & \dr{<1,>10}     & 0.89    & -       & 0.89    & 0.88    & -       & -       \\
                                                   & Random CV: clamp                    &                       &                 & 0.87    & -       & 0.71    & 0.96    & -       & -       \\
    \cmidrule{2-10}
    Moorthy (2021) \cite{Moorthy2021}              & Random CV                           & ML (SVM)              & 10              & 0.78    & -       & 0.65    & 0.90    & 0.71    & 0.57    \\
    \cmidrule{2-10}
    \qr{Wu (2021) \cite{Wu2021}}                   & \qr{Random}                         & \qr{DL (GNN)}         & 1               & -       & -       & -       & -       & 0.69    & -       \\
                                                   &                                     &                       & 5               & -       & -       & -       & -       & 0.63    & -       \\
                                                   &                                     &                       & 10              & -       & -       & -       & -       & 0.67    & -       \\
                                                   &                                     &                       & 30              & -       & -       & -       & -       & 0.65    & -       \\
    \cmidrule{2-10}
    Xiong (2021) \cite{Xiong2021}                  & Stratified Random                   & DL (GNN)              & 10              & 0.89    & -       & 0.87    & 0.91    & 0.94    & 0.78    \\
    \cmidrule{2-10}
    Arab (2022) \cite{Arab2022}                    & Ext                                 & ML (RF)               & 30              & 0.93    & -       & 0.99    & 0.75    & -       & 0.80    \\
    %\cmidrule{2-10}
    \newpage
    \dr{Delre (2022) \cite{Delre2022}}             & \dr{Temporal}                       & \dr{ML (Consensus)}   & 1               & -       & 0.72    & 0.66    & 0.77    & 0.73    & 0.34    \\
                                                   &                                     &                       & 10              & -       & 0.72    & 0.67    & 0.76    & 0.75    & 0.43    \\
    \cmidrule{2-10}
    Ishihara (2022) \cite{Ishihara2022}            & Random                              & ML (Ensemble)         & 10              & 0.87    & -       & 0.90    & 0.83    & 0.94    & 0.73    \\
    \cmidrule{2-10}
    \tr{Kim (2022) \cite{Kim2022}}                 & Scaffold Distance                   & \tr{DL (GNN)}         & \tr{10}         & 0.74    & 0.74    & 0.80    & 0.67    & 0.81    & 0.48    \\
                                                   & Ext \cite{Ryu2020}                  &                       &                 & 0.84    & 0.85    & 0.83    & 0.86    & 0.84    & 0.66    \\
                                                   & Ext \cite{Siramshetty2020}          &                       &                 & 0.80    & 0.78    & 0.73    & 0.82    & 0.85    & 0.52    \\
    \cmidrule{2-10}
    \dr{Krishna (2022) \cite{Krishna2022}}         & Random CV                           & \dr{ML (RF)}          & \dr{-}          & 0.95    & 0.93    & 0.88    & 0.97    & -       & 0.86    \\
                                                   & Ext \cite{PubChem}                  &                       &                 & 0.93    & 0.77    & 0.98    & 0.55    & -       & 0.64    \\
    \cmidrule{2-10}
    \qr{Lanevskij (2022) \cite{Lanevskij2022}}     & Random                              & \qr{ML (XGB)}         & 10              & 0.73    & -       & 0.79    & 0.66    & 0.80    & 0.46    \\
                                                   & \tr{Temporal}                       &                       & 5               & 0.79    & -       & 0.54    & 0.86    & 0.81    & 0.40    \\
                                                   &                                     &                       & 10              & 0.73    & -       & 0.70    & 0.75    & 0.80    & 0.43    \\
                                                   &                                     &                       & 30              & 0.72    & -       & 0.86    & 0.52    & 0.77    & 0.41    \\
    \cmidrule{2-10}
    \tr{Melnikov (2022) \cite{Melnikov2022}}       & \tr{Temporal}                       & \tr{ML (XGB)}         & 5               & 0.73    & 0.70    & 0.51    & 0.88    & -       & -       \\
                                                   &                                     &                       & 10              & 0.77    & 0.79    & 0.69    & 0.90    & -       & -       \\
                                                   &                                     &                       & 30              & 0.82    & 0.75    & 0.51    & 0.88    & -       & -       \\
    \cmidrule{2-10}
    \qr{Shan (2022) \cite{Shan2022}}               & Random CV \cite{Cai2019}            & \qr{DL (GNN)}         & \dr{10}         & -       & -       & -       & -       & 0.96    & -       \\
                                                   & Scaffold CV \cite{Cai2019}          &                       &                 & -       & -       & -       & -       & 0.92    & -       \\
                                                   & Ext \cite{Doddareddy2010}           &                       & <10,>30         & 0.90    & -       & 0.90    & 0.91    & 0.96    & -       \\
                                                   & Ext \cite{Siramshetty2020}          &                       & 30              & 0.80    & -       & 0.81    & 0.80    & 0.86    & -       \\
    \cmidrule{2-10}
    \qr{Zhang (2022) \cite{Zhang2022}}             & Random CV \cite{Cai2019}            & \qr{ML (Ensemble)}    & \tr{10}         & 0.84    & -       & 0.82    & 0.86    & 0.91    & 0.68    \\
                                                   & Ext \cite{Braga2015}                &                       &                 & 0.81    & -       & 0.87    & 0.77    & 0.89    & 0.63    \\
                                                   & Ext \cite{Cai2019}                  &                       &                 & 0.98    & -       & 0.99    & 0.97    & 1.00    & 0.92    \\
                                                   & Ext \cite{Doddareddy2010}           &                       & <10,>30         & 0.92    & -       & 0.92    & 0.92    & 0.96    & -       \\
    \cmidrule{2-10}
    Arab (2023) \cite{Arab2023}                    & Distance                            & DL (DNN)              & 10              & 0.81    & -       & 0.87    & 0.76    & -       & 0.62    \\
    \cmidrule{2-10}
    Banerjee (2023) \cite{Banerjee2023}            & Sorted Y-based                      & ML (RF)               & -               & 0.78    & -       & 0.73    & -       & 0.82    & 0.55    \\
    \cmidrule{2-10}
    \pr{Chen (2023) \cite{Chen2023}}               & \pr{Random}                         & ML (RF)               & 1               & 0.89    & -       & 0.51    & 0.96    & 0.88    & -       \\
                                                   &                                     & \tr{DL (GCN)}         & 10              & 0.80    & -       & 0.79    & 0.82    & 0.88    & -       \\
                                                   &                                     &                       & 30              & 0.88    & -       & 0.95    & 0.61    & 0.90    & -       \\
                                                   &                                     &                       & <1,>10          & 0.89    & -       & 0.80    & 0.93    & 0.93    & -       \\
                                                   &                                     & ML (SVM)              & <10,>30         & 0.89    & -       & 0.97    & 0.65    & 0.92    & -       \\
    \cmidrule{2-10}
    Das (2023) \cite{Das2023}                      & Random                              & ML (Many Tied)        & 10              & 1.00    & -       & 1.00    & 1.00    & 1.00    & 1.00    \\
    \cmidrule{2-10}
    \tr{Feng (2023) \cite{Feng2023}}               & Ext \cite{Doddareddy2010}           & \tr{DL (GNN)}         & <10,>30         & 0.92    & -       & 0.92    & 0.93    & 0.98    & 0.84    \\
                                                   & Ext \cite{Braga2015}                &                       & 10              & 0.81    & -       & 0.80    & 0.83    & 0.89    & 0.63    \\
                                                   & Ext \cite{Cai2019}                  &                       & 10              & 1.00    & -       & 1.00    & 1.00    & 1.00    & 0.99    \\
    \cmidrule{2-10}
    Wang\sep (2023) \cite{Wang2023_CBM}            & Random CV                           & DL (GNN)              & 10              & 0.82    & -       & 0.78    & 0.85    & -       & 0.63    \\
    \cmidrule{2-10}
    Wang\sep (2023) \cite{Wang2023_FiP}            & Random CV \cite{Creanza2021}        & DL (GAT)              & <10,>30         & 0.89\dg & -       & 0.89\dg & -       & 0.89\dg & -       \\
    \cmidrule{2-10}
    \dr{Vittorio (2023) \cite{Vittorio2023}}       & Random CV                           & ML (RF)               & \dr{10}         & 0.79    & -       & 0.84    & 0.72    & 0.87    & 0.57    \\
                                                   & Ext \cite{Doddareddy2010}           & ML (Ensemble)         &                 & 0.69    & -       & 0.79    & 0.43    & 0.67    & 0.22    \\
    %\cmidrule{2-10}
    \newpage
    \dr{Ylipää (2023) \cite{Ylipaa2023}}           & Random CV \cite{Ogura2019}          & \dr{DL (GNN)}         & \dr{10}         & -       & 0.90    & 0.89    & 0.94    & -       & 0.48    \\
                                                   & Random \cite{Ogura2019}             &                       &                 & -       & 0.90    & 0.88    & 0.94    & -       & 0.47    \\
    \cmidrule{2-10}
    Arab (2024) \cite{Arab2024}                    & Distance                            & ML (RF)               & 10              & 0.82    & -       & 0.92    & 0.70    & -       & 0.64    \\
    \cmidrule{2-10}
    Chen (2024) \cite{Chen2024}                    & Random                              & DL (DNN)              & 10              & 0.82    & -       & 0.83    & 0.81    & 0.89    & -       \\
    \cmidrule{2-10}
    He (2024) \cite{He2024}                        & -                                   & DL (CLM)              & -               & 0.80    & -       & 0.80    & 0.80    & 0.88    & 0.61    \\
    \cmidrule{2-10}
    Liu (2024) \cite{Liu2024}                      & Random CV \cite{Karim2021}          & ML/DL (Ensemble)      & 10              & 0.87    & -       & 0.87    & 0.86    & 0.94    & 0.73    \\
    \cmidrule{2-10}
    \dr{Sanches (2024) \cite{Sanches2024}}         & \dr{Random}                         & ML (LightGBM)         & 10              & -       & 0.87    & 0.85    & 0.86    & 0.82    & 0.52    \\
                                                   &                                     & ML (RF)               & <1,<10,>10      & -       & 0.83    & 0.82    & 0.83    & 0.80    & 0.56    \\
    \cmidrule{2-10}
    Wang (2024) \cite{Wang2024}                    & Distance                            & DL (Multimodal)       & 10              & 0.83    & -       & 0.86    & 0.79    & -       & 0.66    \\
    \cmidrule{2-10}
    \tr{Yang (2024) \cite{Yang2024}}               & Scaffold Distance                   & \tr{DL (GNN)}         & 10              & 0.75    & 0.74    & 0.81    & 0.68    & 0.82    & 0.49    \\
                                                   & Ext \cite{Ryu2020}                  &                       & 10              & 0.66    & 0.66    & 0.67    & 0.64    & 0.71    & 0.29    \\
                                                   & Ext \cite{Siramshetty2020}          &                       & 10              & 0.72    & 0.70    & 0.66    & 0.74    & 0.80    & 0.36    \\
    \cmidrule{2-10}
    Cano (2025) \cite{Cano2025}                    & Random \cite{Ogura2019}             & ML (Ensemble)         & 10              & 0.90    & -       & 0.83    & 0.90    & -       & 0.41    \\
    \cmidrule{2-10}
    Feng (2025) \cite{Feng2025}                    & Distance \cite{Arab2023}            & DL (Multimodal)       & 10              & 0.85    & -       & -       & -       & -       & 0.83    \\
    \cmidrule{2-10}
    Gambacorta (2025) \cite{Gambacorta2025}        & Random Nested CV                    & ML (RF)               & 10              & 0.76    & -       & 0.74    & 0.79    & 0.84    & 0.53    \\
	\cmidrule{2-10}
    \dr{Han (2025) \cite{Han2025}}                 & Stratified Random                   & \dr{ML (RF)}          & 10              & 0.80    & -       & 0.80    & 0.80    & 0.88    & 0.60    \\
                                                   & Ext \cite{TDC}                      &                       & 10              & 0.79    & -       & 0.77    & 0.80    & 0.87    & 0.57    \\
	\cmidrule{2-10}
    \dr{Hossain (2025) \cite{Hossain2025}}         & Scaffold                            & \dr{DL (LTN)}         & \dr{10}         & 0.93    & 0.93    & 0.93    & 0.93    & -       & 0.85    \\
                                                   & Ext \cite{Arab2023}                 &                       &                 & 0.83    & 0.83    & 0.78    & 0.90    & -       & 0.66    \\
	\cmidrule{2-10}
    \tr{Jin (2025) \cite{Jin2025}}                 & Random CV \cite{Yang2024}           & \tr{DL (Multimodal)}  & \tr{10}         & 0.85    & 0.83    & 0.85    & 0.81    & 0.91    & 0.68    \\
                                                   & Ext \cite{Ryu2020}                  &                       &                 & 0.80    & 0.77    & 0.83    & 0.71    & 0.85    & 0.54    \\
                                                   & Ext \cite{Siramshetty2020}          &                       &                 & 0.73    & 0.74    & 0.83    & 0.66    & 0.82    & 0.43    \\
	\cmidrule{2-10}
    \dr{Jing (2025) \cite{Jing2025}}               & Random                              & \dr{DL (GNN)}         & \dr{10}         & 0.75    & -       & 0.80    & -       & -       & 0.52    \\
                                                   & Ext \cite{Ryu2020}                  &                       &                 & 0.84    & -       & 0.93    & -       & -       & 0.66    \\
	\cmidrule{2-10}
    Kyro (2025) \cite{Kyro2025}                    & Distance \cite{Arab2023}            & DL (Multimodal)       & 10              & 0.84    & -       & 0.86    & 0.80    & -       & 0.67    \\
	\cmidrule{2-10}
    \tr{Lee (2025) \cite{Lee2025}}                 & Random                              & \tr{DL (GNN,GRU)}     & \tr{10}         & 0.87    & -       & -       & -       & 0.91    & -       \\
                                                   & Ext \cite{Cai2019}                  &                       &                 & 0.84    & -       & -       & -       & 0.87    & -       \\
                                                   & Ext \cite{Karim2021}                &                       &                 & 0.96    & -       & -       & -       & 0.98    & -       \\
	\cmidrule{2-10}
    \dr{Liu\sep (2025) \cite{K_Liu2025}}           & Random                              & \dr{DL (Transformer)} & \dr{10}         & 0.85    & 0.84    & 0.87    & 0.81    & 0.93    & 0.68    \\
                                                   & Ext                                 &                       &                 & 0.86    & 0.86    & 0.90    & 0.82    & 0.93    & 0.72    \\
	\cmidrule{2-10}
    Liu\sep (2025) \cite{L_Liu2025}                & Random CV                           & DL (Multimodal)       & 10              & 0.93    & -       & -       & -       & 0.94    & -       \\
	\cmidrule{2-10}
    \dr{Mohammad (2025) \cite{Mohammad2025}}       & Diverse\cite{Creanza2021}           & ML (XGB)              & <10,>60         & 0.88    & -       & 0.91    & 0.81    & -       & -       \\
                                                   & Ext \cite{Yu2025}                   & ML (RF)               & 10              & 0.84    & -       & 0.84    & 0.86    & -       & -       \\
	\cmidrule{2-10}
    Tran-Nguyen (2025) \cite{TranNguyen2025}       & Scaffold                            & DL (DNN)              & 20              & -       & 0.76    & 0.86    & 0.65    & -       & -       \\
	\cmidrule{2-10}
    \dr{Xu (2025) \cite{Xu2025}}                   & Scaffold \cite{Karim2021}           & \dr{ML (XGB)}         & \dr{10}         & 0.75    & -       & -       & -       & 0.83    & 0.23    \\
                                                   & Ext                                 &                       &                 & 0.76    & -       & 0.79    & 0.73    & -       & 0.52    \\
    %\cmidrule{2-10}
    \dr{Yu (2025) \cite{Yu2025}}                   & Random CV                           & \dr{ML (XGB)}         & \dr{10}         & 0.81    & -       & 0.85    & 0.76    & -       & 0.61    \\
                                                   & Distance                            &                       &                 & 0.73    & -       & 0.80    & 0.62    & -       & 0.41    \\
	\cmidrule{2-10}
    Yang (2025) \cite{Yang2025}                    & Random                              & ML (RF)               & 30              & 0.77    & -       & 0.90    & 0.44    & 0.81    & 0.40    \\
	\cmidrule{2-10}
    Agarwal (2026) \cite{Agarwal2026}              & Random                              & DL (AttentiveFP)      & 10              & -       & -       & -       & -       & 0.89    & -       \\
    \cmidrule{2-10}
    \tr{Beidja (2026) \cite{Beidja2026}}           & Random CV \cite{Wang2023_CBM}       & \tr{DL (Multimodal)}  & \tr{10}         & 0.82    & -       & 0.85    & 0.79    & 0.91    & 0.64    \\
                                                   & Ext \cite{Ryu2020}                  &                       &                 & 0.81    & -       & 0.78    & 0.87    & 0.88    & 0.60    \\
                                                   & Ext \cite{Siramshetty2020}          &                       &                 & 0.76    & -       & 0.83    & 0.73    & 0.87    & 0.50    \\
    \cmidrule{2-10}
    Chu (2026) \cite{Chu2026}                      & Diverse \cite{Wang2016}             & DL (Multimodal)       & 40              & -       & -       & -       & -       & 0.90    & -       \\
    \cmidrule{2-10}
    Enokiya (2026) \cite{Enokiya2026}              & Stratified CV \cite{PubChem}        & DL (GATv2)            & -               & -       & -       & -       & -       & 0.84    & -       \\
    \cmidrule{2-10}
    \dr{Jovanović (2026) \cite{Jovanovi2026}}      & \dr{Distance}                       & \dr{DL (Multimodal)}  & 1               & 0.99    & -       & 0.56    & 0.99    & 0.96    & 0.38    \\
                                                   &                                     &                       & 10              & 0.97    & -       & 0.64    & 0.98    & 0.92    & 0.47    \\
    \cmidrule{2-10}
    \dr{Liang (2026) \cite{Liang2026}}             & Random CV                           & \dr{DL (Multimodal)}  & \dr{10}         & 0.84    & -       & -       & -       & 0.93    & 0.69    \\
                                                   & Stratified Random                   &                       &                 & 0.85\dg & -       & 0.82\dg & 0.88\dg & 0.93\dg & 0.70\dg \\
    \cmidrule{2-10}
    \qr{Su (2026) \cite{Su2026}}                   & Random CV                           & \qr{DL (Multimodal)}  & \qr{10}         & 0.81    & -       & 0.84    & 0.78    & 0.89    & 0.61    \\
                                                   & \dr{Ext \cite{Karim2021}}           &                       &                 & 0.71    & -       & 0.91    & 0.63    & 0.83    & 0.48    \\
                                                   &                                     &                       &                 & 0.89    & -       & 0.93    & 0.79    & 0.92    & 0.73    \\
                                                   & Ext \cite{Siramshetty2018}          &                       &                 & 0.76    & -       & 0.62    & 0.97    & 0.88    & 0.60    \\
    \cmidrule{2-10}
    Sun (2026) \cite{Sun2026}                      & Random CV                           & ML (SVM)              & 10              & 0.80    & -       & 0.78    & 0.83    & 0.88    & -       \\
    \cmidrule{2-10}
    \dr{Zhang (2026) \cite{Zhang2026}}             & Random CV                           & \dr{DL (Multimodal)}  & \dr{10}         & -       & 0.88    & 0.84    & 0.91    & 0.95    & 0.76    \\
                                                   & Scaffold                            &                       &                 & -       & 0.84    & 0.81    & 0.86    & 0.92    & 0.67    \\
	\cmidrule{2-10}
    \dr{Zhao (2026) \cite{Zhao2026}}               & Distance                            & \dr{DL (Multimodal)}  & <1,>10          & -       & -       & 0.75    & -       & 0.83    & -       \\
                                                   & Ext \cite{Arab2023}                 &                       & 10              & 0.83    & -       & 0.79    & 0.89    & -       & 0.67    \\
"""

table_s3_nav1_5 = r"""
    \tr{Khalifa (2020) \cite{Khalifa2020}}  & \tr{Distance}            & \tr{ML (GBM)}        & 1       & 0.93    & - & 0.99    & 0.78    & -       & 0.82    \\
                                            &                          &                      & 10      & 0.93    & - & 0.94    & 0.92    & -       & 0.85    \\
                                            &                          &                      & 30      & 0.94    & - & 0.93    & 0.95    & -       & 0.83    \\
    \cmidrule{2-10}
    Arab (2022) \cite{Arab2022}             & Random CV                & ML (SVM)             & 30      & 0.87    & - & 0.88    & 0.85    & -       & 0.71    \\
    \cmidrule{2-10}
    Kong (2022) \cite{Kong2022}             & Distance                 & ML (RF)              & 30      & 0.93    & - & 0.90    & 0.96    & 0.95    & 0.86    \\
    \cmidrule{2-10}
    Arab (2023) \cite{Arab2023}             & Distance                 & DL (DNN)             & 10      & 0.82    & - & 0.86    & 0.73    & -       & 0.58    \\
    \cmidrule{2-10}
    Arab (2024) \cite{Arab2024}             & Distance                 & ML (RF)              & 10      & 0.86    & - & 0.95    & 0.67    & -       & 0.66    \\
    \cmidrule{2-10}
    Chen (2024) \cite{Chen2024}             & Random                   & DL (DNN)             & 10      & 0.86    & - & 0.90    & 0.78    & 0.89    & -       \\
    \cmidrule{2-10}
    Wang (2024) \cite{Wang2024}             & Distance                 & DL (Multimodal)      & 10      & 0.86    & - & -       & -       & 0.92    & -       \\
    \cmidrule{2-10}
    Feng (2025) \cite{Feng2025}             & Distance \cite{Arab2023} & DL (Multimodal)      & 10      & 0.87    & - & -       & -       & -       & 0.79    \\
    \cmidrule{2-10}
    Gambacorta (2025) \cite{Gambacorta2025} & Random Nested CV         & ML (RF)              & 10      & 0.80    & - & 0.77    & 0.83    & 0.89    & 0.61    \\
    \cmidrule{2-10}
    Kyro (2025) \cite{Kyro2025}             & Distance \cite{Arab2023} & DL (Multimodal)      & 10      & 0.89    & - & 0.96    & 0.76    & -       & 0.75    \\
    \newpage
    Agarwal (2026) \cite{Agarwal2026}       & Random                   & DL (AttentiveFP)     & 10      & -       & - & -       & -       & 0.95    & -       \\
    \cmidrule{2-10}
    Jovanović (2026) \cite{Jovanovi2026}    & Distance                 & DL (Multimodal)      & 10      & 0.76    & - & 0.70    & 0.78    & 0.83    & 0.45    \\
    \cmidrule{2-10}
    \dr{Liang (2026) \cite{Liang2026}}      & Random CV                & \dr{DL (Multimodal)} & \dr{10} & 0.86    & - & -       & -       & 0.92    & 0.67    \\
                                            & Stratified Random        &                      &         & 0.86\dg & - & 0.89\dg & 0.82\dg & 0.94\dg & 0.70\dg \\
"""

table_s3_cav1_2 = r"""
    Arab (2023) \cite{Arab2023}             & Distance                 & DL (DNN)             & 10      & 0.86    & - & 0.96    & 0.69    & -       & 0.70    \\
	\cmidrule{2-10}
    Arab (2024) \cite{Arab2024}             & Distance                 & ML (RF)              & 10      & 0.91    & - & 0.98    & 0.79    & -       & 0.81    \\
    \cmidrule{2-10}
    Chen (2024) \cite{Chen2024}             & Random                   & DL (DNN)             & 10      & 0.86    & - & 0.87    & 0.85    & 0.94    & -       \\
    \cmidrule{2-10}
    Wang (2024) \cite{Wang2024}             & Distance                 & DL (Multimodal)      & 10      & 0.86    & - & -       & -       & 0.93    & -       \\
    \cmidrule{2-10}
    Feng (2025) \cite{Feng2025}             & Distance \cite{Arab2023} & DL (Multimodal)      & 10      & 0.86    & - & -       & -       & -       & 0.78    \\
    \cmidrule{2-10}
    Gambacorta (2025) \cite{Gambacorta2025} & Random Nested CV         & ML (XGB)             & 10      & 0.79    & - & 0.81    & 0.76    & 0.88    & 0.57    \\
    \cmidrule{2-10}
    Kyro (2025) \cite{Kyro2025}             & Distance \cite{Arab2023} & DL (Multimodal)      & 10      & 0.91    & - & 0.96    & 0.83    & -       & 0.81    \\
    \cmidrule{2-10}
    Agarwal (2026) \cite{Agarwal2026}       & Random                   & DL (GIN)             & 10      & -       & - & -       & -       & 0.93    & -       \\
    \cmidrule{2-10}
    Jovanović (2026) \cite{Jovanovi2026}    & Distance                 & DL (Multimodal)      & 10      & 0.78    & - & 0.70    & 0.81    & 0.84    & 0.49    \\
    \cmidrule{2-10}
    \dr{Liang (2026) \cite{Liang2026}}      & Random CV                & \dr{DL (Multimodal)} & \dr{10} & 0.87    & - & -       & -       & 0.94    & 0.69    \\
                                            & Stratified Random        &                      &         & 0.87\dg & - & 0.95\dg & 0.80\dg & 0.95\dg & 0.72\dg \\
"""

table_s3_kv7_1 = r"""
    Jovanović (2026) \cite{Jovanovi2026} & Distance & DL (Multimodal) & 10 & 0.86 & - & 0.00 & 0.92 & 0.38 & -0.08 \\
"""

In [27]:
table_s4_herg = r"""
    \dr{Cavalli (2002) \cite{Cavalli2002}}        & LOO CV                       & \dr{PH/ST (PLS)}   & 0.77    & -       & -        \\
                                                  & Manual                       &                    & 0.74    & -       & -        \\
	\cmidrule{2-6}
    Keserü (2003) \cite{Keser2003}                & Stratified Random            & ST (MLR)           & 0.81    & -       & -        \\
    \cmidrule{2-6}
    Pearlstein (2003) \cite{Pearlstein2003}       & LOO CV                       & PH/ST (PLS)        & 0.57    & -       & -        \\
	\cmidrule{2-6}
    Aptula (2004) \cite{Aptula2004}               & Sorted Uniform               & ST (MLR)           & 0.95\ddg& 0.43\ddg& 0.41\ddg \\
	\cmidrule{2-6}
    Coi (2006) \cite{Coi2006}                     & Random                       & ST (MLR)           & 0.82    & -       & 0.41     \\
	\cmidrule{2-6}
    Ekins (2006) \cite{Ekins2006}                 & -                            & ML (DT)            & 0.33    & -       & -        \\
	\cmidrule{2-6}
    Seierstad (2006) \cite{Seierstad2006}         & Random CV                    & DL (Ensemble)      & 0.51\ddg& 0.85\ddg& 0.71\ddg \\
    \cmidrule{2-6}
    Song (2006) \cite{Song2006}                   & Random                       & ML (SVM)           & 0.85    & 0.60    & -        \\
	\cmidrule{2-6}
    Yoshida (2006) \cite{Yoshida2006}             & Random CV                    & ST (MLR)           & 0.66    & -       & -        \\
	\cmidrule{2-6}
    Du (2007) \cite{Du2007}                       & Random CV                    & ST (LR)            & 0.56    & -       & -        \\
	\cmidrule{2-6}
    Leong (2007) \cite{Leong2007}                 & Manual                       & PH/ML (SVM)        & 0.94    & -       & -        \\
	\cmidrule{2-6}
    Obrezanova (2007) \cite{Obrezanova2007}       & Sorted Uniform               & ML (GP)            & 0.63    & 0.46    & -        \\
	\cmidrule{2-6}
    Gunturi (2008) \cite{Gunturi2008}             & Random CV \cite{Yoshida2006} & ST (LLR)           & 0.82    & -       & -        \\
	\cmidrule{2-6}
    Kramer (2008) \cite{Kramer2008}               & Random                       & ML (SVM)           & 0.70    & 0.74    & 0.55     \\
	\cmidrule{2-6}
    \dr{Thai\sep (2008) \cite{Thai2008_CBDD}}     & Random                       & \dr{DL (CPNN)}     & 0.87    & -       & -        \\
                                                  & Distance                     &                    & 0.87    & -       & -        \\
    \cmidrule{2-6}
    Coi (2009) \cite{Coi2009}                     & DISE                         & ST (MLR)           & 0.86    & -       & -        \\
	\cmidrule{2-6}
    \dr{Ermondi (2009) \cite{Ermondi2009}}        & LOO CV                       & \dr{PH/ST (PLS)}   & 0.69\ddg& 0.37\ddg& -        \\
                                                  & Ext \cite{Cavalli2002}       &                    & 0.19\ddg& 0.90\ddg& 0.81\ddg \\
	\cmidrule{2-6}
    Hansen (2009) \cite{Hansen2009}               & LCO CV                       & ML (Ensemble)      & 0.54    & 0.57    & -        \\
	\cmidrule{2-6}
    Cuny (2011) \cite{Cuny2011}                   & Distance                     & ST (PLS)           & 0.59    & -       & -        \\
	\cmidrule{2-6}
    \dr{Obiol-Pardo (2011) \cite{ObiolPardo2011}} & LOO CV                       & \dr{PH/ST (PLS)}   & 0.46    & 0.92    & -        \\
                                                  & Ext                          &                    & -       & 0.89    & -        \\
	\cmidrule{2-6}
    Sinha (2011) \cite{Sinha2011}                 & Stratified Random            & ST (MLR)           & 0.64    & 0.78    & 0.68     \\
	\cmidrule{2-6}
    \dr{Kar (2012) \cite{Kar2012}}                & Random                       & ST (GFA)           & 0.60    & -       & -        \\
                                                  & Ext                          & ST (PLS)           & 0.60    & -       & -        \\
	\cmidrule{2-6}
    Tan (2012) \cite{Tan2012}                     & Distance                     & ML (Ensemble)      & 0.75    & -       & 0.45     \\
	\cmidrule{2-6}
    Gobbi (2016) \cite{Gobbi2016}                 & Random CV                    & ST (MLR)           & 0.93    & -       & -        \\
	\cmidrule{2-6}
    Sheridan (2016) \cite{Sheridan2016}           & Temporal                     & -                  & -       & -       & -        \\
	\cmidrule{2-6}
    \dr{Chemi (2017) \cite{Chemi2017}}            & Random CV                    & \dr{PH/ST (PLS)}   & 0.80    & -       & -        \\
                                                  & Ext                          &                    & 0.86    & -       & -        \\
	%\cmidrule{2-6}
    \dr{Munawar (2018) \cite{Munawar2018}}        & LOO CV                       & \dr{PH/ST (PLS)}   & 0.63    & 0.84    & -        \\
                                                  & Diverse                      &                    & 0.60    & -       & -        \\
	\cmidrule{2-6}
    \dr{Wacker (2018) \cite{Wacker2018}}          & Random CV                    & \dr{ML (XGB)}      & 0.6\s   & -       & -        \\
                                                  & Ext \cite{CiPA_promiscuity}  &                    & 0.68    & 0.71    & 0.52     \\
	\cmidrule{2-6}
    \dr{Munawar (2019) \cite{Munawar2019}}        & Diverse                      & \dr{PH/ST (PLS)}   & 0.58    & -       & -        \\
                                                  & Ext                          &                    & 0.51    & -       & -        \\
	\cmidrule{2-6}
    \dr{Lee (2021) \cite{Lee2021}}                & Random CV: binding           & \dr{ML (XGB)}      & 0.76    & 0.58    & -        \\
                                                  & Random CV: clamp             &                    & 0.80    & 0.59    & -        \\
    \cmidrule{2-6}
    \tr{Meng (2021) \cite{Meng2021}}              & Random CV                    & \tr{ML (SVM)}      & -       & 0.59\dg & -        \\
                                                  & Ext \cite{Konda2019}         &                    & -       & 0.88    & -        \\
                                                  & Ext \cite{Munawar2019}       &                    & -       & 1.11    & -        \\
    \cmidrule{2-6}
    Sato (2021) \cite{Sato2021}                   & Random CV                    & ML (SVM)           & 0.59    & 0.60    & -        \\
	\cmidrule{2-6}
    Stergiopoulos (2021) \cite{Stergio2021}       & Random                       & ST (MLR)           & 0.72    & -       & -        \\
    \cmidrule{2-6}
    Arab (2022) \cite{Arab2022}                   & Ext                          & ML (RF)            & 0.67    & 0.77    & -        \\
    \cmidrule{2-6}
    \tr{Ding (2022) \cite{Ding2022}}              & Random Nested CV             & \tr{ML (Ensemble)} & 0.65    & 0.53    & 0.36     \\
                                                  & Ext \cite{Konda2019}         &                    & 0.49    & 0.76    & 0.46     \\
                                                  & Ext \cite{Negami2019}        &                    & 0.62    & 0.77    & 0.50     \\
	\cmidrule{2-6}
    \dr{Lanevskij (2022) \cite{Lanevskij2022}}    & Random: patch-clamp          & \dr{ML (XGB)}      & 0.41\s  & 0.55\s  & 0.44\s   \\
                                                  & Random: all assays           &                    & 0.29    & 0.63    & 0.49     \\
    \cmidrule{2-6}
    Melnikov (2022) \cite{Melnikov2022}           & Temporal                     & ML (XGB)           & 0.46    & 0.59    & 0.47     \\
	\cmidrule{2-6}
    Banerjee (2023) \cite{Banerjee2023}           & Sorted Y-based               & ST (PLS)           & -       & -       & 0.55     \\
	\cmidrule{2-6}
    Das (2023) \cite{Das2023}                     & -                            & ST (MLR)           & -       & 0.47    & 0.38     \\
	\cmidrule{2-6}
    Sanches (2024) \cite{Sanches2024}             & Random                       & ML (SVM)           & 0.61    & 0.44    & 0.35     \\
	\cmidrule{2-6}
    Kyro (2025) \cite{Kyro2025}                   & Distance \cite{Arab2023}     & DL (Multimodal)    & 0.44    & 0.74    & 0.51     \\
	\cmidrule{2-6}
    Liu\sep (2025) \cite{L_Liu2025}               & Random CV                    & DL (Multimodal)    & 0.68    & 0.45    & -        \\
    \cmidrule{2-6}
    \dr{Yu (2025) \cite{Yu2025}}                  & Random CV                    & \dr{ML (XGB)}      & 0.62    & 0.60    & 0.38     \\
                                                  & Distance                     &                    & 0.36    & 0.53    & 0.40     \\
    \cmidrule{2-6}
    Zhang (2025) \cite{Zhang2025}                 & Random                       & ML (XGB)           & 0.78    & -       & -        \\
    \cmidrule{2-6}
    Jovanović (2026) \cite{Jovanovi2026}          & Distance                     & DL (Multimodal)    & -       & 0.72    & -        \\
    \cmidrule{2-6}
    \dr{Sun (2026) \cite{Sun2026}}                & Random CV                    & \dr{ML (SVM)}      & 0.63    & 0.55    & 0.38     \\
                                                  & Temporal                     &                    & -       & -       & 0.50     \\
"""

table_s4_nav1_5 = r"""
    Khalifa (2020) \cite{Khalifa2020}    & Distance                 & ML (RF)         & 0.71 & 0.73 & -    \\
    \cmidrule{2-6}
    Arab (2022) \cite{Arab2022}          & Ext                      & ML (RF)         & 0.71 & 0.51 & -    \\
    \cmidrule{2-6}
    Kyro (2025) \cite{Kyro2025}          & Distance \cite{Arab2023} & DL (Multimodal) & 0.32 & 0.62 & 0.43 \\
    \cmidrule{2-6}
    Jovanović (2026) \cite{Jovanovi2026} & Distance                 & DL (Multimodal) & -    & 0.57 & -    \\
"""

table_s4_cav1_2 = r"""
    Wiśniowska (2012) \cite{Winiowska2012} & Random CV                & DN (Ensemble)   & 0.29 & 1.10 & -    \\
    \cmidrule{2-6}
    Kyro (2025) \cite{Kyro2025}            & Distance \cite{Arab2023} & DL (Multimodal) & 0.54 & 1.01 & 0.82 \\
    \cmidrule{2-6}
    Jovanović (2026) \cite{Jovanovi2026}   & Distance                 & DL (Multimodal) & -    & 0.82 & -    \\
"""

table_s4_kv7_1 = r"""
    \dr{Obiol-Pardo (2011) \cite{ObiolPardo2011}} & LOO CV    & \dr{PH/ST (PLS)} & 0.41 & 0.99 & - \\
                                                  & Ext       &                  & -    & 0.82 & - \\
    \cmidrule{2-6}
    Polak (2012) \cite{Polak2012}                 & Random CV & DL (Ensemble)    & -    & 0.86 & - \\
"""

In [28]:
table_s7 = r"""
    Coi (2009) \cite{Coi2009}                 & Descriptor range                           & -                                      \\
    \cmidrule{2-3}
    Nisius (2009) \cite{Nisius2009JCIM}       & Distance to training set                   & -                                      \\
    \cmidrule{2-3}
    Obrezanova (2010) \cite{Obrezanova2010}   & -                                          & Gaussian process posterior uncertainty \\
    \cmidrule{2-3}
    Sinha (2011) \cite{Sinha2011}             & Leverage                                   & -                                      \\
    \cmidrule{2-3}
    Kar (2012) \cite{Kar2012}                 & Distance to training set                   & -                                      \\
    \cmidrule{2-3}
    Kirevaa (2013) \cite{Kireeva2013}         & Prediction-confidence-based AD             & Prediction probability/confidence      \\
    \cmidrule{2-3}
    Braga (2014) \cite{Braga2014}             & Distance to training set                   & -                                      \\
    \cmidrule{2-3}
    Gobbi (2016) \cite{Gobbi2016}             & Fragment-based defect                      & -                                      \\
    \cmidrule{2-3}
    Alves (2018) \cite{Alves2018}             & Leverage                                   & -                                      \\
    \cmidrule{2-3}
    Munawar (2018) \cite{Munawar2018}         & Distance to training set                   & -                                      \\
    \cmidrule{2-3}
    Siramshetty (2018) \cite{Siramshetty2018} & Distance to training set                   & -                                      \\
    \cmidrule{2-3}
    Wacker (2018) \cite{Wacker2018}           & Distance to training set                   & -                                      \\
    \cmidrule{2-3}
    Yang (2018) \cite{Yang2018}               & Descriptor range                           & -                                      \\
    \cmidrule{2-3}
    Ogura (2019) \cite{Ogura2019}             & Distance to training set                   & -                                      \\
    \cmidrule{2-3}
    Lee (2021) \cite{Lee2021}                 & Distance to training set                   & -                                      \\
    \cmidrule{2-3}
    Delre (2022) \cite{Delre2022}             & Leverage                                   & -                                      \\
    \cmidrule{2-3}
    Kim (2022) \cite{Kim2022}                 & -                                          & Monte Carlo dropout                    \\
    \cmidrule{2-3}
    Krishna (2022) \cite{Krishna2022}         & Distance to training set                   & -                                      \\
    \cmidrule{2-3}
    Banerjee (2023) \cite{Banerjee2023}       & Response-discordance AD                    & -                                      \\
    \cmidrule{2-3}
    Chen (2023) \cite{Chen2023}               & Distance to training set                   & -                                      \\
    \cmidrule{2-3}
    Wang (2023) \cite{Wang2023_FiP}           & Leverage                                   & -                                      \\
    \cmidrule{2-3}
    Vittorio (2023) \cite{Vittorio2023}       & Consensus AD                               & Ensemble variance                      \\
    \cmidrule{2-3}
    Arab (2024) \cite{Arab2024}               & -                                          & Ensemble prediction confidence         \\
    \cmidrule{2-3}
    Liu (2024) \cite{Liu2024}                 & Descriptor range; distance to training set & -                                      \\
    \cmidrule{2-3}
    Sanches (2024) \cite{Sanches2024}         & Leverage                                   & -                                      \\
    \cmidrule{2-3}
    Cano (2025) \cite{Cano2025}               & Distance to training set                   & -                                      \\
    \cmidrule{2-3}
    Gambacorta (2025) \cite{Gambacorta2025}   & Prediction-score reliability AD            & -                                      \\
    \cmidrule{2-3}
    Yu (2025) \cite{Yu2025}                   & Distance to training set                   & -                                      \\
    \cmidrule{2-3}
    Zhang (2025) \cite{Zhang2025}             & Similarity-activity landscape              & -                                      \\
    \cmidrule{2-3}
    Enokiya (2026) \cite{Enokiya2026}         & Distance to training set                   & Monte Carlo dropout                    \\
    %\newpage
    Jovanović (2026) \cite{Jovanovi2026}      & Distance to training set                   & -                                      \\
    \cmidrule{2-3}
    Zhang (2026) \cite{Zhang2026}             & Similarity-activity landscape              & -                                      \\
"""

In [29]:
table_s8 = r"""
	 Ponti (2001) \cite{Ponti2001}             & Yes          & No   & No         & No   & Data                                 & No \\
     \cmidrule{2-7}
     Cavalli (2002) \cite{Cavalli2002}         & Yes          & No   & Yes        & No   & Data, Pharmacophore                  & No \\
	 \cmidrule{2-7}
     Ekins (2002) \cite{Ekins2002}             & Yes          & No   & Yes        & No   & Data, Pharmacophore                  & No \\
	 \cmidrule{2-7}
     Roche (2002) \cite{Roche2002}             & Partially    & No   & No         & No   & Data\dg                              & No \\
	 \cmidrule{2-7}
     Ekins (2003) \cite{Ekins2003}             & No           & No   & No         & No   & No                                   & No \\
	 \cmidrule{2-7}
     Keserü (2003) \cite{Keser2003}            & Yes          & No   & Yes        & No   & Data, Equation                       & No \\
	 \cmidrule{2-7}
     Pearlstein (2003) \cite{Pearlstein2003}   & Yes          & No   & Partially  & No   & Data, Interactions                   & No \\
	 \cmidrule{2-7}
     Aronov (2004) \cite{Aronov2004}           & No           & No   & Yes        & No   & Pharmacophore                        & No \\
     \cmidrule{2-7}
     Aptula (2004) \cite{Aptula2004}           & Yes          & No   & Yes        & No   & Data, Equation                       & No \\
	 \cmidrule{2-7}
     Bains (2004) \cite{Bains2004}             & Yes          & No   & Yes        & No   & Data, Descriptors, Pharmacophore     & No \\
	 \cmidrule{2-7}
     Fioravanzo (2004) \cite{Fioravanzo2004}   & Yes (SI)     & No   & No         & No   & Data                                 & No \\
	 \cmidrule{2-7}
     Cianchetta (2005) \cite{Cianchetta2005}   & Yes (SI)     & No   & Partially  & No   & Data\dg, Pharmacophore               & No \\
	 \cmidrule{2-7}
     O'Brien (2005) \cite{OBrien2005}          & No           & No   & No         & No   & No                                   & No \\
	 \cmidrule{2-7}
     Tobita (2005) \cite{Tobita2005}           & Yes          & No   & No         & No   & Data, Descriptors                    & No \\
	 \cmidrule{2-7}
     Aronov (2006) \cite{Aronov2006}           & No           & No   & Yes        & No   & Pharmacophore                        & No \\
	 \cmidrule{2-7}
     Coi (2006) \cite{Coi2006}                 & Yes          & No   & Yes        & No   & Data, Equation                       & No \\
	 \cmidrule{2-7}
     Dubus (2006) \cite{Dubus2006}             & No           & No   & Yes        & No   & Descriptors, Decision tree           & No \\
	 \cmidrule{2-7}
     Ekins (2006) \cite{Ekins2006}             & Yes          & No   & No         & No   & Data, Descriptors                    & No \\
	 \cmidrule{2-7}
     Gepp (2006) \cite{Gepp2006}               & No           & No   & Yes        & No   & Decision tree, Descriptors           & No \\
	 \cmidrule{2-7}
     Seierstad (2006) \cite{Seierstad2006}     & Partially    & No   & Partially  & No   & Data\dg, Descriptors                 & No \\
	 \cmidrule{2-7}
     Sun (2006) \cite{Sun2006}                 & Partially    & No   & Partially  & No   & Data\dg, Descriptors                 & No \\
	 \cmidrule{2-7}
     Song (2006) \cite{Song2006}               & Partially    & No   & Partially  & No   & Data\dg, Fragments                   & No \\
	 \cmidrule{2-7}
     Yoshida (2006) \cite{Yoshida2006}         & Yes          & No   & Yes        & No   & Data, Equation                       & No \\
	 \cmidrule{2-7}
     Du (2007) \cite{Du2007}                   & Yes          & No   & Yes        & No   & Data, Equation                       & No \\
	 \cmidrule{2-7}
     Leong (2007) \cite{Leong2007}             & Yes          & No   & Partially  & No   & Data, Pharmacophore                  & No \\
	 \cmidrule{2-7}
     Obrezanova (2007) \cite{Obrezanova2007}   & Yes (SI)     & No   & No         & No   & No                                   & No \\
	 \cmidrule{2-7}
     Chekmarev (2008) \cite{Chekmarev2008}     & Yes (SI)     & No   & No         & No   & No                                   & No \\
	 \cmidrule{2-7}
     Gunturi (2008) \cite{Gunturi2008}         & Yes          & No   & Partially  & No   & Data, Descriptors                    & No \\
	 \cmidrule{2-7}
     Jia (2008) \cite{Jia2008}                 & Partially    & No   & No         & No   & Data\dg                              & No \\
	 \cmidrule{2-7}
     Kramer (2008) \cite{Kramer2008}           & Yes (SI)     & No   & Partially  & No   & Pharmacophore                        & No \\
	 %\cmidrule{2-7}
     \newpage
     Li (2008) \cite{Li2008}                   & Yes (SI)     & No   & No         & No   & No                                   & No \\
	 \cmidrule{2-7}
     Thai\sep (2008) \cite{Thai2008BMC}        & Yes (SI)     & No   & No         & No   & Descriptors                          & No \\
	 \cmidrule{2-7}
     Thai\sep (2008) \cite{Thai2008_CBDD}      & Yes (SI)     & No   & No         & No   & Descriptors                          & No \\
	 \cmidrule{2-7}
     Wang (2008) \cite{Wang2008}               & No           & No   & No         & No   & Descriptors                          & No \\
	 \cmidrule{2-7}
     Coi (2009) \cite{Coi2009}                 & Yes          & No   & Yes        & No   & Data, Equation                       & No \\
	 \cmidrule{2-7}
     Ermondi (2009) \cite{Ermondi2009}         & Yes          & No   & No         & No   & Data                                 & No \\
	 \cmidrule{2-7}
     Hansen (2009) \cite{Hansen2009}           & No           & No   & No         & No   & No                                   & No \\
	 \cmidrule{2-7}
     Nisius\sep (2009) \cite{Nisius2009CBDD}   & Yes (SI)     & No   & Partially  & No   & No                                   & No \\
	 \cmidrule{2-7}
     Nisius\sep (2009) \cite{Nisius2009JCIM}   & Yes (SI)     & No   & No         & No   & No                                   & No \\
	 \cmidrule{2-7}
     Thai (2009) \cite{Thai2009}               & Yes (SI)     & No   & No         & No   & No                                   & No \\
	 \cmidrule{2-7}
     Doddareddy (2010) \cite{Doddareddy2010}   & Yes (SI)     & No   & Partially  & No   & Fragments, Decision tree             & No \\
	 \cmidrule{2-7}
     Obrezanova (2010) \cite{Obrezanova2010}   & Yes (SI)     & No   & No         & No   & No                                   & No \\
	 \cmidrule{2-7}
     Su (2010) \cite{Su2010}                   & Yes (SI)     & No   & Partially  & No   & Equation                             & No \\
	 \cmidrule{2-7}
     Wiśniowska (2010) \cite{Wisniowska2010}   & No           & No   & No         & No   & Descriptors                          & No \\
	 \cmidrule{2-7}
     Cuny (2011) \cite{Cuny2011}               & No           & No   & Partially  & No   & Descriptors, Pharmacophore, Equation & No \\
	 \cmidrule{2-7}
     Kim (2011) \cite{Kim2011}                 & No           & No   & No         & No   & Fragments                            & No \\
	 \cmidrule{2-7}
     Obiol-Pardo (2011) \cite{ObiolPardo2011}  & Yes (SI)     & No   & Partially  & No   & No                                   & No \\
	 \cmidrule{2-7}
     Robinson (2011) \cite{Robinson2011}       & Yes (SI)     & No   & No         & No   & Descriptors                          & No \\
	 \cmidrule{2-7}
     Sinha (2011) \cite{Sinha2011}             & Yes          & No   & No         & No   & Data                                 & No \\
	 \cmidrule{2-7}
     Shen (2011) \cite{Shen2011}               & No           & No   & Partially  & No   & Descriptors                          & No \\
	 \cmidrule{2-7}
     Broccatelli (2012) \cite{Broccatelli2012} & Yes (SI)     & No   & Partially  & No   & No                                   & No \\
     \cmidrule{2-7}
     Kar (2012) \cite{Kar2012}                 & Yes (SI)     & No   & Yes        & No   & Equation                             & No \\
     \cmidrule{2-7}
     Polak (2012) \cite{Polak2012}             & Yes          & No   & No         & No   & Data, Hyperparameters                & No \\
	 \cmidrule{2-7}
     Tan (2012) \cite{Tan2012}                 & Yes          & No   & Partially  & No   & Data, Equation                       & No \\
	 \cmidrule{2-7}
     Wang (2012) \cite{Wang2012}               & Yes          & No   & Yes        & No   & Decision tree, Fragments             & \href{https://cadd.suda.edu.cn/admet}{Data}\p \\
     \cmidrule{2-7}
     Wiśniowska (2012) \cite{Winiowska2012}    & Yes          & No   & No         & No   & Data, Hyperparameters                & No \\
	 \cmidrule{2-7}
     Czodrowski (2013) \cite{Czodrowski2013}   & Yes (SI)     & Yes  & No         & No   & Fragments                            & \href{https://github.com/pzc/herg_chembl_jcim}{Code} \\
	 \cmidrule{2-7}
     Kireeva (2013) \cite{Kireeva2013}         & No           & No   & No         & No   & No                                   & No \\
	 \cmidrule{2-7}
     Braga (2014) \cite{Braga2014}             & Yes          & No   & No         & Yes  & No                                   & \href{https://labmol.farmacia.ufg.br/predherg/vs-wdi.pdf}{Data}\p; \href{https://labmol.farmacia.ufg.br/predherg}{Platform}\p \\
	 \cmidrule{2-7}
     Kratz (2014) \cite{Kratz2014}             & Partially    & No   & Yes        & No   & Data, Pharmacophore                  & No \\
	 \cmidrule{2-7}
     Liu (2014) \cite{Liu2014}                 & No           & No   & No         & No   & Fragments                            & No \\
	 \cmidrule{2-7}
     Braga (2015) \cite{Braga2015}             & Yes (Online) & No   & No         & Yes  & No                                   & \href{https://labmol.farmacia.ufg.br/predherg}{Platform}\p \\
	 %\cmidrule{2-7}
     \newpage
     Du (2015) \cite{Du2015}                   & Yes (SI)     & No   & No         & No   & No                                   & No \\
	 \cmidrule{2-7}
     Ma (2015) \cite{Ma2015}                   & No           & No   & Partially  & No   & Hyperparameters                      & No \\
	 \cmidrule{2-7}
     Chavan (2016) \cite{Chavan2016}           & Yes (SI)     & No   & No         & No   & No                                   & No \\
	 \cmidrule{2-7}
     Didziapetris (2016) \cite{Didzia2016}     & Yes (SI)     & No   & No         & No   & No                                   & No \\
	 \cmidrule{2-7}
     Gobbi (2016) \cite{Gobbi2016}             & Yes          & No   & Partially  & No   & Equation                             & No \\
	 \cmidrule{2-7}
     Sheridan (2016) \cite{Sheridan2016}       & No           & No   & No         & No   & No                                   & No \\
	 \cmidrule{2-7}
     Wang (2016) \cite{Wang2016}               & Yes (SI)     & No   & Yes        & No   & Decision tree, Pharmacophore         & No \\
	 \cmidrule{2-7}
     Zhang (2016) \cite{Zhang2016}             & Yes (SI)     & No   & No         & No   & Fragments                            & No \\
	 \cmidrule{2-7}
     Chemi (2017) \cite{Chemi2017}             & Yes (SI)     & No   & No         & No   & Pharmacophore                        & No \\
	 \cmidrule{2-7}
     Li (2017) \cite{Li2017}                   & Yes          & No   & No         & Yes  & No                                   & \href{https://ochem.eu/article/103592}{Data, Model} \\
	 \cmidrule{2-7}
     Sun (2017) \cite{Sun2017}                 & Yes (SI)     & No   & No         & No   & Pharmacophore                        & No \\
	 \cmidrule{2-7}
     Alves (2018) \cite{Alves2018}             & Yes          & No   & No         & Yes  & No                                   & \href{https://chembench.mml.unc.edu/mudra}{Code}$^{*}$ \\
	 \cmidrule{2-7}
     Munawar (2018) \cite{Munawar2018}         & Yes (SI)     & No   & No         & No   & Interactions, Pharmacophore          & No \\
	 \cmidrule{2-7}
     Siramshetty (2018) \cite{Siramshetty2018} & Yes          & Yes  & No         & No   & No                                   & \href{https://github.com/AGPreissner/Publications}{Data, Code} \\
	 \cmidrule{2-7}
     Wacker (2018) \cite{Wacker2018}           & No           & No   & No         & No   & No                                   & No \\
	 \cmidrule{2-7}
     Yang (2018) \cite{Yang2018}               & No           & No   & No         & Yes  & No                                   & \href{https://lmmd.ecust.edu.cn/admetsar2}{Platform} \\
	 \cmidrule{2-7}
     Cai (2019) \cite{Cai2019}                 & Yes (SI)     & Yes  & No         & No   & No                                   & \href{https://github.com/ChengF-Lab/deephERG}{Code} \\
	 \cmidrule{2-7}
     Hu (2019) \cite{Hu2019}                   & No           & No   & No         & No   & No                                   & No \\
	 \cmidrule{2-7}
     Konda (2019) \cite{Konda2019}             & Yes (SI)     & No   & No         & No   & No                                   & No \\
	 \cmidrule{2-7}
     Lee (2019) \cite{Lee2019}                 & On Request   & No   & No         & Yes  & Features                             & \href{http://bioanalysis.cau.ac.kr:7050}{Platform} \\
	 \cmidrule{2-7}
     Munawar (2019) \cite{Munawar2019}         & No           & No   & No         & No   & Interactions                         & No \\
	 \cmidrule{2-7}
     Negami (2019) \cite{Negami2019}           & Yes (SI)     & No   & No         & No   & No                                   & No \\
	 \cmidrule{2-7}
     Ogura (2019) \cite{Ogura2019}             & Yes          & No   & No         & Yes  & No                                   & \href{https://drugdesign.riken.jp/hERGdb/}{Platform} \\
	 \cmidrule{2-7}
     Zhang (2019) \cite{Zhang2019}             & Yes (SI)     & No   & No         & No   & No                                   & No \\
	 \cmidrule{2-7}
     Choi (2020) \cite{Choi2020}               & Yes (SI)     & No   & No         & No   & No                                   & No \\
	 \cmidrule{2-7}
     Khalifa (2020) \cite{Khalifa2020}         & Yes (SI)     & No   & No         & No   & No                                   & No \\
	 \cmidrule{2-7}
     Kim (2020) \cite{Kim2020}                 & No           & No   & No         & No   & No                                   & No \\
	 \cmidrule{2-7}
     Liu (2020) \cite{Liu2020}                 & No           & No   & No         & No   & Fragments                            & No \\
	 \cmidrule{2-7}
     Ryu (2020) \cite{Ryu2020}                 & Partially    & No   & Yes        & No   & No                                   & \href{https://bitbucket.org/krictai/deephit}{Platform}\p \\
	 \cmidrule{2-7}
     Siramshetty (2020) \cite{Siramshetty2020} & Yes          & Yes  & Yes        & No   & No                                   & \href{https://github.com/ncats/herg-ml}{Data, Code, Model} \\
	 \cmidrule{2-7}
     Wang (2020) \cite{Wang2020}               & On Request   & No   & No         & No   & Algorithm, Parameters                & No \\
	 \cmidrule{2-7}
     Creanza (2021) \cite{Creanza2021}         & Yes (SI)     & No   & No         & No   & Interactions, Protein                & No \\
	 %\cmidrule{2-7}
     \newpage
     Karim (2021) \cite{Karim2021}             & Yes          & Yes  & No         & No   & No                                   & \href{https://github.com/Abdulk084/CardioTox}{Data, Code} \\
	 \cmidrule{2-7}
     Lee (2021) \cite{Lee2021}                 & Yes          & Yes  & No         & No   & No                                   & \href{https://github.com/NIDA-IRP-CCMB/QSAR_DAT-hERG}{Data, Code} \\
	 \cmidrule{2-7}
     Meng (2021) \cite{Meng2021}               & Yes (SI)     & No   & No         & No   & No                                   & No \\
	 \cmidrule{2-7}
     Moorthy (2021) \cite{Moorthy2021}         & Yes (SI)     & No   & Yes        & No   & Fragments, Equation                  & No \\
	 \cmidrule{2-7}
     Sato (2021) \cite{Sato2021}               & No           & No   & No         & No   & No                                   & \href{https://drugdesign.riken.jp/hERG}{Platform}\p \\
     \cmidrule{2-7}
     Stergiopoulos (2021) \cite{Stergio2021}   & Yes          & No   & Yes        & No   & Data, Experimental, Equation         & No \\
	 \cmidrule{2-7}
     Wu (2021) \cite{Wu2021}                   & Yes          & Yes  & Yes        & No   & No                                   & \href{https://github.com/wzxxxx/MGA}{Data, Code, Model} \\
	 \cmidrule{2-7}
     Xiong (2021) \cite{Xiong2021}             & No           & No   & No         & Yes  & No                                   & \href{https://admetmesh.scbdd.com}{Platform} \\
	 \cmidrule{2-7}
     Arab (2022) \cite{Arab2022}               & Yes          & Yes  & Yes        & No   & No                                   & \href{https://github.com/issararab/ToxTree}{Data, Code, Model} \\
	 \cmidrule{2-7}
     Delre (2022) \cite{Delre2022}             & Yes          & Yes  & Yes        & No   & No                                   & \href{https://github.com/PDelre93/hERG-QSAR}{Data, Code, Workflow} \\
	 \cmidrule{2-7}
     Ding (2022) \cite{Ding2022}               & Yes (SI)     & Yes  & Yes        & No   & No                                   & \href{https://github.com/Liu-Lab-Lnu/MDFP-hERG/tree/main/MDFP-hERG}{Data, Code, Model} \\
	 \cmidrule{2-7}
     Ishihara (2022) \cite{Ishihara2022}       & No           & No   & No         & No   & No                                   & No \\
	 \cmidrule{2-7}
     Kim (2022) \cite{Kim2022}                 & Yes          & Yes  & Yes        & No   & No                                   & \href{https://github.com/GIST-CSBL/BayeshERG}{Data, Code, Model} \\
     \cmidrule{2-7}
     Kong (2022) \cite{Kong2022}               & No           & No   & No         & No   & No                                   & No \\
	 \cmidrule{2-7}
     Krishna (2022) \cite{Krishna2022}         & Yes (SI)     & Yes  & No         & No   & No                                   & \href{https://github.com/ABorrel/cardiotox_hERG}{Code} \\
	 \cmidrule{2-7}
     Lanevskij (2022) \cite{Lanevskij2022}     & Yes (SI)     & No   & No         & No   & No                                   & No \\
	 \cmidrule{2-7}
     Melnikov (2022) \cite{Melnikov2022}       & No           & No   & No         & No   & No                                   & No \\
	 \cmidrule{2-7}
     Shan (2022) \cite{Shan2022}               & Yes          & Yes  & No         & No   & No                                   & \href{https://github.com/AI-amateur/DMPNN-hERG}{Data, Code} \\
	 \cmidrule{2-7}
     Zhang (2022) \cite{Zhang2022}             & Yes (Online) & No   & No         & Yes  & Fragments                            & \href{http://www.icdrug.com/ICDrug/T}{Platform}\p \\
	 \cmidrule{2-7}
     Arab (2023) \cite{Arab2023}               & Yes          & Yes  & Yes        & No   & No                                   & \href{https://github.com/issararab/CToxPred}{Data, Code, Model} \\
	 \cmidrule{2-7}
     Banerjee (2023) \cite{Banerjee2023}       & Yes (SI)     & No   & Yes        & No   & No                                   & \href{https://sites.google.com/jadavpuruniversity.in/dtc-lab-software/home}{Model} \\
	 \cmidrule{2-7}
     Chen (2023) \cite{Chen2023}               & Yes (SI)     & No   & No         & No   & Fragments                            & No \\
	 \cmidrule{2-7}
     Das (2023) \cite{Das2023}                 & Yes (SI)     & No   & Yes        & No   & Equation                             & No \\
	 \cmidrule{2-7}
     Feng (2023) \cite{Feng2023}               & Yes          & Yes  & No         & No   & No                                   & \href{https://github.com/WeilabMSU/hERG-prediction}{Data, Code} \\
	 \cmidrule{2-7}
     Wang\sep (2023) \cite{Wang2023_CBM}       & Yes          & Yes  & No         & No   & No                                   & \href{https://github.com/zhaoqi106/DMFGAM}{Data, Code} \\
	 \cmidrule{2-7}
     Wang\sep (2023) \cite{Wang2023_FiP}       & Yes          & Yes  & No         & No   & No                                   & \href{https://github.com/huijia-wang/hERG_ChEMBL240}{Data, Code} \\
	 \cmidrule{2-7}
     Vittorio (2023) \cite{Vittorio2023}       & Yes          & No   & Yes        & No   & No                                   & \href{https://zenodo.org/records/7551783}{Data, Model} \\
	 \cmidrule{2-7}
     Ylipää (2023) \cite{Ylipaa2023}           & No           & No   & No         & No   & No                                   & No \\
	 \cmidrule{2-7}
     Arab (2024) \cite{Arab2024}               & Yes          & Yes  & Yes        & No   & No                                   & \href{https://github.com/issararab/CToxPred2}{Data, Code, Model} \\
	 \cmidrule{2-7}
     Chen (2024) \cite{Chen2024}               & No           & No   & No         & Yes  & Fragments                            & \href{http://cardiodpi.sapredictor.cn}{Platform} \\
	 \cmidrule{2-7}
     He (2024) \cite{He2024}                   & Partially    & Yes  & Yes        & No   & No                                   & \href{https://github.com/heshida01/CLOP-hERG}{Data, Code}; \href{https://drive.google.com/drive/folders/1ysH7cOSYr8ARBZ3BiKVGDDGmaIYyD3pP}{Model} \\
	 \cmidrule{2-7}
     Liu (2024) \cite{Liu2024}                 & No           & No   & No         & No   & No                                   & No \\
	 %\cmidrule{2-7}
     \newpage
     Sanches (2024) \cite{Sanches2024}         & Yes (SI)     & Yes  & No         & Yes  & No                                   & \href{https://github.com/LabMolUFG/Pred_hERG}{Data, Code}; \href{https://predherg.labmol.com.br}{Platform} \\
	 \cmidrule{2-7}
     Wang (2024) \cite{Wang2024}               & Yes          & Yes  & No         & Yes  & No                                   & \href{https://github.com/taowang11/MultiCBlo}{Data, Code}; \href{https://huggingface.co/spaces/wtttt/PCICB}{Platform} \\
	 \cmidrule{2-7}
     Yang (2024) \cite{Yang2024}               & Yes          & Yes  & Yes        & No   & No                                   & \href{https://github.com/Tianbiao-Yang/AttenhERG}{Data, Code, Model} \\
	 \cmidrule{2-7}
     Cano (2025) \cite{Cano2025}               & No           & No   & No         & Yes  & No                                   & \href{https://www.hypercubane.re}{Platform} \\
	 \cmidrule{2-7}
     Feng (2025) \cite{Feng2025}               & Yes          & Yes  & Yes        & No   & No                                   & \href{https://github.com/3505675604/MultiCTox}{Data, Code}; \href{https://huggingface.co/seyonec/PubChem10M_SMILES_BPE_450k}{Model} \\
	 \cmidrule{2-7}
     Gambacorta (2025) \cite{Gambacorta2025}   & Yes          & Yes  & No         & Yes  & No                                   & \href{https://github.com/f48r1/cupid}{Data, Code}; \href{https://prometheus.farmacia.uniba.it/cupid/}{Platform}\\
	 \cmidrule{2-7}
     Han (2025) \cite{Han2025}                 & Yes          & Yes  & Yes        & No   & No                                   & \href{https://github.com/CADD-SC/ADMET_Prediction_Models}{Data, Code, Model} \\
	 \cmidrule{2-7}
     Hossain (2025) \cite{Hossain2025}         & Yes          & Yes  & Yes        & No   & No                                   & \href{https://github.com/hossain013/hERG-LTN}{Data, Code, Model} \\
	 \cmidrule{2-7}
     Jin (2025) \cite{Jin2025}                 & No           & Yes  & No         & No   & No                                   & \href{https://github.com/zhaoqi106/hERG-MFFGNN}{Code} \\
	 \cmidrule{2-7}
     Jing (2025) \cite{Jing2025}               & No           & No   & No         & No   & No                                   & No \\
	 \cmidrule{2-7}
     Kyro (2025) \cite{Kyro2025}               & Yes          & Yes  & Yes        & No   & No                                   & \href{https://github.com/gregory-kyro/CardioGenAI}{Data, Code, Model} \\
	 \cmidrule{2-7}
     Lee (2025) \cite{Lee2025}                 & Yes          & Yes  & No         & No   & No                                   & \href{https://github.com/bmil-jnu/hERGAT}{Data, Code} \\
	 \cmidrule{2-7}
     Liu (2025) \cite{L_Liu2025}               & No           & No   & No         & No   & No                                   & No \\
	 \cmidrule{2-7}
     Liu (2025) \cite{K_Liu2025}               & No           & Yes  & No         & No   & No                                   & \href{https://github.com/liuliwei1980/MTF-hERG}{Code} \\
	 \cmidrule{2-7}
     Mohammad (2025) \cite{Mohammad2025}       & Yes          & Yes  & No         & No   & No                                   & \href{https://github.com/syedmohs/hERG-toxicity-prediction}{Data, Code} \\
	 \cmidrule{2-7}
     Tran-Nguyen (2025) \cite{TranNguyen2025}  & Yes          & Yes  & No         & No   & No                                   & \href{https://github.com/vktrannguyen/HERGAI}{Data, Code} \\
	 \cmidrule{2-7}
     Xu (2025) \cite{Xu2025}                   & Yes (SI)     & No   & No         & No   & No                                   & No \\
	 \cmidrule{2-7}
     Yang (2025) \cite{Yang2025}               & No           & Yes  & No         & No   & No                                   & \href{https://github.com/liu-sheng00/Machine-learning-prediction-model}{Code} \\
     \cmidrule{2-7}
     Yu (2025) \cite{Yu2025}                   & Yes          & No   & No         & Yes  & No                                   & \href{http://ssbio.cau.ac.kr/software/hergboost}{Data,Platform} \\
	 \cmidrule{2-7}
     Zhang (2025) \cite{Zhang2025}             & Yes          & Yes  & Yes        & No   & No                                   & \href{https://github.com/ZYX2222/hERG_binding_regressor}{Data, Code, Model} \\
     \cmidrule{2-7}
     Agarwal (2026) \cite{Agarwal2026}         & Yes          & Yes  & Yes        & No   & No                                   & \href{https://github.com/pip700/cardiotox_prediction}{Data, Code, Model} \\
     \cmidrule{2-7}
     Beidja (2026) \cite{Beidja2026}           & Yes          & Yes  & No         & No   & No                                   & \href{https://github.com/BeidjaCheikh/TDMFLSGAT_data_and_code}{Data, Code} \\
	 \cmidrule{2-7}
     Chu (2026) \cite{Chu2026}                 & Yes          & Yes  & No         & No   & No                                   & \href{https://github.com/cjw0206/MEMOL}{Data, Code} \\
     \cmidrule{2-7}
     Enokiya (2026) \cite{Enokiya2026}         & No           & No   & No         & No   & No                                   & Data, Code\ddg \\
     \cmidrule{2-7}
     Jovanović (2026) \cite{Jovanovi2026}      & Yes          & Yes  & Yes        & Yes  & No                                   & \href{https://github.com/AppliedScientific/CardioSafe-benchmark/}{Data, Code, Model}; \href{https://platform.appliedscientific.ai/cardiosafe}{Platform} \\
     \cmidrule{2-7}
     Liang (2026) \cite{Liang2026}             & No           & No   & No         & No   & No                                   & \href{http://cardiactox.cqudfbp.net:88/}{Platform}\p \\
     \cmidrule{2-7}
     Su (2026) \cite{Su2026}                   & No (request) & No   & No         & No   & No                                   & No \\
     \cmidrule{2-7}
     Sun (2026) \cite{Sun2026}                 & Yes (SI)     & No   & No         & No   & No                                   & No \\
     \cmidrule{2-7}
     Zhang (2026) \cite{Zhang2026}             & Yes (SI)     & Yes  & Yes        & No   & No                                   & \href{https://github.com/ZYX2222/hERG_Blockers_Classification}{Data, Code, Model} \\
	 \cmidrule{2-7}
     Zhao (2026) \cite{Zhao2026}               & Yes          & Yes  & No         & No   & No                                   & \href{https://github.com/ConfusedAnt/FEAOF}{Data, Code} \\
"""

# Conversion

In [30]:
s1_columns = "Study & Study type & Models & Features & Channel".split(" & ")

df_s1 = latex_2_df(table_s1, s1_columns)

Cavalli (2002) \cite{Cavalli2002}         & PH, Reg      & PLS                                             & 3D Shape-based & $hERG$ 
Ekins (2002) \cite{Ekins2002}             & PH           & Alignment-based                                 & 3D Shape-based (CATALYST) & $hERG$ 
Roche (2002) \cite{Roche2002}             & BC           & SOM, PCA, PLS, NN                               & MD (VolSurf, DRAGON) & $hERG$
Ekins (2003) \cite{Ekins2003}             & PH           & Alignment-based                                 & 3D Shape-based (CATALYST) & $hERG$ 
Keserü (2003) \cite{Keser2003}            & Reg          & LR + hQSAR                                      & MD (VolSurf, Sybyl) & $hERG$
Pearlstein (2003) \cite{Pearlstein2003}   & PH           & 3D Shape-based                                  & Pharmacophore & $hERG$ 
Aptula (2004) \cite{Aptula2004}           & Reg          & MLR                                             & MD, Quantum Chemical & $hERG$ 
Aronov (2004) \cite{Aronov

In [31]:
s2_columns = r"Study & Size & No. Toxic & Data Type & Availability & Sources".split(" & ")

df_s2 = pl.concat([
    latex_2_df(table_s2_herg, s2_columns).with_columns(pl.lit("hERG").alias("Channel")),
    latex_2_df(table_s2_nav1_5, s2_columns).with_columns(pl.lit("Nav1.5").alias("Channel")),
    latex_2_df(table_s2_cav1_2, s2_columns).with_columns(pl.lit("Cav1.2").alias("Channel")),
    latex_2_df(table_s2_kv7_1, s2_columns).with_columns(pl.lit("Kv7.1").alias("Channel"))
])

Ponti (2001) \cite{Ponti2001}                 & 137    & N/A         & IC50       & In paper\s\dg & Literature 
Cavalli (2002) \cite{Cavalli2002}             & 37     & N/A         & IC50       & In paper      & Study \cite{Ponti2001}, FenichelDB 
Roche (2002) \cite{Roche2002}                 & 472    & N/A         & IC50       & In paper\dg   & In-house Roche, known drugs 
Ekins (2002) \cite{Ekins2002}                 & 22     & N/A         & IC50       & In paper      & In-house ElliLilly 
Keserü (2003) \cite{Keser2003}                & 68     & N/A         & IC50       & In paper      & Study \cite{Roche2002}, FenichelDB
Pearlstein (2003) \cite{Pearlstein2003}       & 23     & N/A         & IC50       & In paper      & In-house Aventis Pharma 
Aronov (2004) \cite{Aronov2004}               & 414    & 85@40\uM    & Binary     & Online        & Studies \cite{Cavalli2002, Ekins2002}, FenichelDB, Literature 
Aptula (2004) \cite{Aptula2004}               & 19     & N/A         & IC50     

In [32]:
s3_columns = r"Study & Splitting/Evaluation & Model & Threshold & ACC & BA & SEN & SPE & ROC AUC & MCC".split(" & ")

df_s3 = pl.concat([
    latex_2_df(table_s3_herg, s3_columns).with_columns(pl.lit("hERG").alias("Channel")),
    latex_2_df(table_s3_nav1_5, s3_columns).with_columns(pl.lit("Nav1.5").alias("Channel")),
    latex_2_df(table_s3_cav1_2, s3_columns).with_columns(pl.lit("Cav1.2").alias("Channel")),
    latex_2_df(table_s3_kv7_1, s3_columns).with_columns(pl.lit("Kv7.1").alias("Channel"))
])

In [33]:
s4_columns = r"Study & Splitting/Evaluation & Model & R2 & RMSE & MAE".split(" & ")

df_s4 = pl.concat([
    latex_2_df(table_s4_herg, s4_columns).with_columns(pl.lit("hERG").alias("Channel")),
    latex_2_df(table_s4_nav1_5, s4_columns).with_columns(pl.lit("Nav1.5").alias("Channel")),
    latex_2_df(table_s4_cav1_2, s4_columns).with_columns(pl.lit("Cav1.2").alias("Channel")),
    latex_2_df(table_s4_kv7_1, s4_columns).with_columns(pl.lit("Kv7.1").alias("Channel"))
])

\dr{Cavalli (2002) \cite{Cavalli2002}}        & LOO CV                       & \dr{PH/ST (PLS)}   & 0.77    & -       & -        
& Manual                       &                    & 0.74    & -       & -        
Keserü (2003) \cite{Keser2003}                & Stratified Random            & ST (MLR)           & 0.81    & -       & -        
Pearlstein (2003) \cite{Pearlstein2003}       & LOO CV                       & PH/ST (PLS)        & 0.57    & -       & -        
Aptula (2004) \cite{Aptula2004}               & Sorted Uniform               & ST (MLR)           & 0.95\ddg& 0.43\ddg& 0.41\ddg 
Coi (2006) \cite{Coi2006}                     & Random                       & ST (MLR)           & 0.82    & -       & 0.41     
Ekins (2006) \cite{Ekins2006}                 & -                            & ML (DT)            & 0.33    & -       & -        
Seierstad (2006) \cite{Seierstad2006}         & Random CV                    & DL (Ensemble)      & 0.51\ddg& 0.85\ddg& 0.71\ddg 
Song (

In [34]:
s7_columns = r"Study & Applicability Domain & Uncertainty Quantification".split(" & ")

df_s7 = latex_2_df(table_s7, s7_columns)

Coi (2009) \cite{Coi2009}                 & Descriptor range                           & -                                      
Nisius (2009) \cite{Nisius2009JCIM}       & Distance to training set                   & -                                      
Obrezanova (2010) \cite{Obrezanova2010}   & -                                          & Gaussian process posterior uncertainty 
Sinha (2011) \cite{Sinha2011}             & Leverage                                   & -                                      
Kar (2012) \cite{Kar2012}                 & Distance to training set                   & -                                      
Kirevaa (2013) \cite{Kireeva2013}         & Prediction-confidence-based AD             & Prediction probability/confidence      
Braga (2014) \cite{Braga2014}             & Distance to training set                   & -                                      
Gobbi (2016) \cite{Gobbi2016}             & Fragment-based defect                      & -       

In [35]:
s8_columns = r"Study & Data & Code & Models & Platform & In-paper & Ref".split(" & ")

df_s8 = latex_2_df(table_s8, s8_columns)

Ponti (2001) \cite{Ponti2001}             & Yes          & No   & No         & No   & Data                                 & No 
Cavalli (2002) \cite{Cavalli2002}         & Yes          & No   & Yes        & No   & Data, Pharmacophore                  & No 
Ekins (2002) \cite{Ekins2002}             & Yes          & No   & Yes        & No   & Data, Pharmacophore                  & No 
Roche (2002) \cite{Roche2002}             & Partially    & No   & No         & No   & Data\dg                              & No 
Ekins (2003) \cite{Ekins2003}             & No           & No   & No         & No   & No                                   & No 
Keserü (2003) \cite{Keser2003}            & Yes          & No   & Yes        & No   & Data, Equation                       & No 
Pearlstein (2003) \cite{Pearlstein2003}   & Yes          & No   & Partially  & No   & Data, Interactions                   & No 
Aronov (2004) \cite{Aronov2004}           & No           & No   & Yes        & No   & Pharmacopho

# Cleaning

In [36]:
df_s1 = df_s1.with_columns(
    pl.col("Channel").str.replace_all(r"\$|_", "").alias("Channel")
)

df_s2 = strip_polyrows(df_s2, columns=["Study", "Size", "No. Toxic", "Data Type", "Availability", "Sources"])
df_s3 = strip_polyrows(df_s3, columns=["Study", "Splitting/Evaluation", "Model", r"Threshold"])
df_s4 = strip_polyrows(df_s4, columns=["Study", "Splitting/Evaluation", "Model"])
df_s7 = strip_polyrows(df_s7, columns=["Study", "Applicability Domain", "Uncertainty Quantification"])
df_s8 = strip_polyrows(df_s8, columns=["Study", "Data", "Code", "Models", "Platform", "In-paper", "Ref"])

In [301]:
df_s1 = df_s1.with_columns(
    pl.all().replace("-", None)
)

df_s1 = df_s1.with_columns(
    pl.col("Study").map_elements(
        lambda study: int(re.search(r"(\(\d{4}\))", study).group(0)[1:-1]),
        return_dtype=pl.Int64
    ).alias("Year")
)

df_s2 = df_s2.with_columns(
    pl.all().replace("-", None)
)

In [302]:
s3_num_cols = ["ACC", "BA", "SEN", "SPE", "ROC AUC", "MCC"]

df_s3 = strip_values(df_s3, columns=s3_num_cols).with_columns([
    pl.all().replace("-", None)
])

# In one entry the ROC AUC is given as lower bound; since we need the column in numerical format, we drop it
df_s3 = df_s3.with_columns(
    pl.when(
        pl.col("ROC AUC").str.contains(">")
    ).then(
        pl.lit(None)
    ).otherwise(
        pl.col("ROC AUC")
    ).alias("ROC AUC")
)

df_s3 = df_s3.cast(
    {col: pl.Float64 for col in s3_num_cols}
)

df_s3 = df_s3.with_columns(
    pl.col("Study").map_elements(
        lambda study: int(re.search(r"(\(\d{4}\))", study).group(0)[1:-1]),
        return_dtype=pl.Int64
    ).alias("Year")
)

In [303]:
s4_num_cols = ["R2", "RMSE", "MAE"]

df_s4 = strip_values(df_s4, columns=s4_num_cols).with_columns([
    pl.all().replace("-", None)
])

df_s4 = df_s4.cast(
    {col: pl.Float64 for col in s4_num_cols}
)

df_s4 = df_s4.with_columns(
    pl.col("Study").map_elements(
        lambda study: int(re.search(r"(\(\d{4}\))", study).group(0)[1:-1]),
        return_dtype=pl.Int64
    ).alias("Year")
)

In [37]:
df_s7 = df_s7.with_columns(
    pl.all().replace("-", None)
)

df_s8 = df_s8.with_columns(
    pl.all().replace("-", None)
)

In [304]:
write_pl(df_s1, "../data/supporting_tables/s1.xlsx")
write_pl(df_s2, "../data/supporting_tables/s2.xlsx")
write_pl(df_s3, "../data/supporting_tables/s3.xlsx")
write_pl(df_s4, "../data/supporting_tables/s4.xlsx")
write_pl(df_s7, "../data/supporting_tables/s7.xlsx")
write_pl(df_s8, "../data/supporting_tables/s8.xlsx")